# Dataset Preparation

## Function Definitions & EXECUTION

In [2]:


import os
import wfdb
import numpy as np
import pandas as pd
from collections import Counter


# CONFIGURATION


data_path = "Path to the dataset"

window = 180

output_path = "Path to output"


# AAMI CLASS MAPPING


AAMI_map = {
    # N class
    'N': 'N',
    'L': 'N',
    'R': 'N',
    'e': 'N',
    'j': 'N',

    # S class
    'A': 'S',
    'a': 'S',
    'J': 'S',
    'S': 'S',

    # V class
    'V': 'V',
    'E': 'V'
}

valid_classes = ['N', 'S', 'V']



# FUNCTION : IDENTIFY RECORDS


def get_records(data_path):
    """
    Identify and sort all ECG recording files
    in the specified dataset directory.
    """

    records = sorted([
        f.split(".")[0]
        for f in os.listdir(data_path)
        if f.endswith(".dat")
    ])

    print("Number of records:", len(records))
    print("Records:", records)

    return records



# FUNCTION : EXTRACT HEARTBEAT SEGMENTS


def extract_heartbeats(records, data_path, window,
                       AAMI_map, valid_classes):
    """
    Extract ECG heartbeat segments centered around
    annotated R-peaks and assign their AAMI labels.
    """

    X = []
    y = []

    for record in records:

        record_path = os.path.join(data_path, record)

        # Read ECG signal
        signal = wfdb.rdrecord(record_path)

        # Read annotations
        annotation = wfdb.rdann(record_path, 'atr')

        # Use first ECG channel
        ecg = signal.p_signal[:, 0]

        for i, sym in enumerate(annotation.symbol):

            if sym in AAMI_map:

                label = AAMI_map[sym]

                if label in valid_classes:

                    # R-peak position
                    pos = annotation.sample[i]

                    # Define heartbeat window
                    start = pos - window
                    end = pos + window

                    # Keep only complete segments
                    if start >= 0 and end <= len(ecg):

                        beat = ecg[start:end]

                        # Ensure exactly 360 samples
                        if len(beat) == 360:

                            X.append(beat)
                            y.append(label)

    return X, y



# FUNCTION : CONVERT TO NUMPY ARRAYS


def convert_to_arrays(X, y):
    """
    Convert extracted heartbeat segments and labels
    into NumPy arrays.
    """

    X = np.array(X, dtype=np.float32)
    y = np.array(y)

    print("\nDataset shape:", X.shape)
    print("Label distribution:", Counter(y))

    return X, y


# FUNCTION : SAVE DATASET


def save_dataset(X, y, output_path):
    """
    Combine heartbeat segments and labels into a
    DataFrame and export the dataset as a CSV file.
    """

    df = pd.DataFrame(X)
    df['label'] = y

    df.to_csv(
        output_path,
        index=False
    )

    print("\nDataset saved:")
    print(output_path)

In [ ]:
# Identify ECG records
records = get_records(data_path)

# Extract heartbeat segments
X, y = extract_heartbeats(
    records,
    data_path,
    window,
    AAMI_map,
    valid_classes
)

# Convert to NumPy arrays
X, y = convert_to_arrays(X, y)

# Save prepared dataset
save_dataset(X, y, output_path)

# Data Preparation for Model Training

## Function Definitions & Execution 

In [5]:

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


# CONFIGURATION

input_path = "Path to output"

output_dir = "path to working"


# FUNCTION : LOAD DATASET

def load_dataset(input_path):
    """
    Check the availability of the input dataset,
    load the CSV file, and display its information.
    """

    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Dataset not found:\n{input_path}\n"
            "Please run Cell 1 first."
        )

    print("Input dataset found:")
    print(input_path)

    df = pd.read_csv(input_path)

    print("DATASET INFORMATION")

    print("Dataset shape:", df.shape)

    print("\nFirst 5 rows:")
    print(df.head())

    return df


# FUNCTION : SEPARATE FEATURES AND LABELS

def separate_features_labels(df):
    """
    Separate ECG features from their corresponding
    categorical class labels.
    """

    X = df.drop("label", axis=1).values
    y = df["label"].values

    print("\nFeatures shape:", X.shape)
    print("Labels shape:", y.shape)

    return X, y


# FUNCTION : ENCODE LABELS

def encode_labels(y):
    """
    Convert categorical class labels into numerical
    representations using LabelEncoder.
    """

    le = LabelEncoder()

    y_encoded = le.fit_transform(y)

    print("LABEL ENCODING")

    print("Classes:", le.classes_)

    for class_name, class_id in zip(
        le.classes_,
        range(len(le.classes_))
    ):
        print(f"{class_name} -> {class_id}")

    return y_encoded, le


# FUNCTION : TRAIN / TEST SPLIT

def split_dataset(X, y_encoded):
    """
    Split the dataset into training and testing
    subsets using stratified heartbeat-level splitting.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.20,
        random_state=42,
        stratify=y_encoded
    )

    print("TRAIN / TEST SPLIT")

    print("Split type: Heartbeat-level stratified split")
    print("Test size: 20%")
    print("Random state: 42")

    print("\nTrain shape:", X_train.shape)
    print("Test shape :", X_test.shape)

    print("\nTrain labels:", y_train.shape)
    print("Test labels :", y_test.shape)

    return X_train, X_test, y_train, y_test


# FUNCTION : STANDARDIZE DATA

def standardize_data(X_train, X_test):
    """
    Apply StandardScaler to the training and testing
    datasets, fitting the scaler only on training data.
    """

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)

    print("STANDARDIZATION")

    print("Standardization completed.")
    print("Scaler fitted only on training data.")
    print("Test data transformed using the training scaler.")

    return X_train, X_test, scaler


# FUNCTION : DISPLAY CLASS DISTRIBUTION

def display_class_distribution(y_train, y_test):
    """
    Calculate and display class distributions and
    percentages for training and testing datasets.
    """

    train_counts = np.bincount(
        y_train,
        minlength=3
    )

    test_counts = np.bincount(
        y_test,
        minlength=3
    )

    print("CLASS DISTRIBUTION")

    print("\nTraining distribution:")
    print(f"N: {train_counts[0]}")
    print(f"S: {train_counts[1]}")
    print(f"V: {train_counts[2]}")

    print("\nTest distribution:")
    print(f"N: {test_counts[0]}")
    print(f"S: {test_counts[1]}")
    print(f"V: {test_counts[2]}")

    # Calculate class percentages
    train_percent = (
        train_counts / len(y_train)
    ) * 100

    test_percent = (
        test_counts / len(y_test)
    ) * 100

    print("\nTraining percentages:")
    print(f"N: {train_percent[0]:.2f}%")
    print(f"S: {train_percent[1]:.2f}%")
    print(f"V: {train_percent[2]:.2f}%")

    print("\nTest percentages:")
    print(f"N: {test_percent[0]:.2f}%")
    print(f"S: {test_percent[1]:.2f}%")
    print(f"V: {test_percent[2]:.2f}%")

    return train_counts, test_counts


# FUNCTION : VERIFY DATA SIZES

def verify_data_sizes(X_train, X_test):
    """
    Display the total number of samples and verify
    the training and testing proportions.
    """

    print("DATA SIZE VERIFICATION")

    print(
        "Total samples:",
        len(X_train) + len(X_test)
    )

    print(
        "Training samples:",
        len(X_train)
    )

    print(
        "Test samples:",
        len(X_test)
    )

    print(
        "Training percentage:",
        len(X_train) / (len(X_train) + len(X_test)) * 100
    )

    print(
        "Test percentage:",
        len(X_test) / (len(X_train) + len(X_test)) * 100
    )


# FUNCTION : SAVE NUMPY ARRAYS

def save_numpy_arrays(
    X_train, X_test, y_train, y_test, le, output_dir
):
    """
    Save the processed training and testing arrays,
    together with the encoded class names.
    """

    np.save(
        os.path.join(output_dir, "X_train.npy"),
        X_train
    )

    np.save(
        os.path.join(output_dir, "X_test.npy"),
        X_test
    )

    np.save(
        os.path.join(output_dir, "y_train.npy"),
        y_train
    )

    np.save(
        os.path.join(output_dir, "y_test.npy"),
        y_test
    )

    np.save(
        os.path.join(output_dir, "classes.npy"),
        le.classes_
    )

    print("FILES SAVED SUCCESSFULLY")

    print(os.path.join(output_dir, "X_train.npy"))
    print(os.path.join(output_dir, "X_test.npy"))
    print(os.path.join(output_dir, "y_train.npy"))
    print(os.path.join(output_dir, "y_test.npy"))
    print(os.path.join(output_dir, "classes.npy"))



In [ ]:


# Load dataset
df = load_dataset(input_path)

# Separate features and labels
X, y = separate_features_labels(df)

# Encode class labels
y_encoded, le = encode_labels(y)

# Perform stratified train/test split
X_train, X_test, y_train, y_test = split_dataset(
    X,
    y_encoded
)

# Apply standardization
X_train, X_test, scaler = standardize_data(
    X_train,
    X_test
)

# Display class distributions
train_counts, test_counts = display_class_distribution(
    y_train,
    y_test
)

# Verify dataset sizes
verify_data_sizes(
    X_train,
    X_test
)

# Save processed arrays
save_numpy_arrays(
    X_train,
    X_test,
    y_train,
    y_test,
    le,
    output_dir
)

# Random Forest

In [6]:
# RANDOM FOREST BASELINE
# FUNCTION DEFINITIONS


import numpy as np

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)




class_names = ["N", "S", "V"]


# FUNCTION : LOAD DATA

def load_model_data():
    """
    Load the prepared training and testing datasets
    from NumPy files.
    """

    X_train = np.load("X_train.npy")
    X_test = np.load("X_test.npy")
    y_train = np.load("y_train.npy")
    y_test = np.load("y_test.npy")

    print("Data loaded successfully.")
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape:", y_test.shape)

    return X_train, X_test, y_train, y_test


# FUNCTION : TRAIN RANDOM FOREST

def train_random_forest(X_train, y_train):
    """
    Initialize and train the Random Forest classifier
    using the specified configuration.
    """

    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    rf.fit(X_train, y_train)

    return rf


# FUNCTION : PREDICTION

def predict_random_forest(rf, X_test):
    """
    Generate predictions using the trained
    Random Forest classifier.
    """

    y_pred = rf.predict(X_test)

    return y_pred


# FUNCTION : EVALUATION METRICS

def evaluate_random_forest(y_test, y_pred):
    """
    Calculate and display the main evaluation
    metrics for the Random Forest classifier.
    """

    accuracy = accuracy_score(y_test, y_pred)

    precision_macro = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1_weighted = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print("\nRANDOM FOREST RESULTS ")

    print(f"Accuracy:          {accuracy:.6f}")
    print(f"Macro Precision:   {precision_macro:.6f}")
    print(f"Macro Recall:      {recall_macro:.6f}")
    print(f"Macro F1-score:    {f1_macro:.6f}")
    print(f"Weighted F1-score: {f1_weighted:.6f}")

    return {
        "Accuracy": accuracy,
        "Macro Precision": precision_macro,
        "Macro Recall": recall_macro,
        "Macro F1-score": f1_macro,
        "Weighted F1-score": f1_weighted
    }


# FUNCTION : CLASSIFICATION REPORT

def display_classification_report(y_test, y_pred):
    """
    Generate and display the classification report
    for the three AAMI classes.
    """

    print("\nCLASSIFICATION REPORT")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1, 2],
            target_names=class_names,
            digits=6,
            zero_division=0
        )
    )


# FUNCTION : CONFUSION MATRIX

def display_confusion_matrix(y_test, y_pred):
    """
    Calculate and display the confusion matrix
    using the specified class order.
    """

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2]
    )

    print("\nCONFUSION MATRIX")
    print("Class order:", class_names)
    print(cm)

    return cm

In [ ]:
# MODEL EXECUTION

# 1. Load data
X_train, X_test, y_train, y_test = load_model_data()


# 2. Train Random Forest
rf = train_random_forest(X_train, y_train)


# 3. Prediction
y_pred = predict_random_forest(rf, X_test)


# 4. Evaluation metrics
rf_results = evaluate_random_forest(
    y_test,
    y_pred
)


# 5. Classification report
display_classification_report(
    y_test,
    y_pred
)


# 6. Confusion matrix
cm = display_confusion_matrix(
    y_test,
    y_pred
)

# XGBoost

In [8]:
# XGBOOST 
# FUNCTION DEFINITIONS

import numpy as np

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


class_names = ["N", "S", "V"]


# FUNCTION : TRAIN XGBOOST

def train_xgboost(X_train, y_train):
    """
    Initialize and train the XGBoost classifier
    using the specified experimental configuration.
    """

    print("\nTraining XGBoost...")

    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="mlogloss",
        random_state=42
    )

    xgb.fit(X_train, y_train)

    return xgb


# FUNCTION : PREDICTION

def predict_xgboost(xgb, X_test):
    """
    Generate predictions using the trained
    XGBoost classifier.
    """

    y_pred = xgb.predict(X_test)

    return y_pred


# FUNCTION : EVALUATION METRICS

def evaluate_xgboost(y_test, y_pred):
    """
    Calculate and display the main evaluation
    metrics for the XGBoost classifier.
    """

    accuracy = accuracy_score(y_test, y_pred)

    precision_macro = precision_score(
        y_test,
        y_pred,
        average='macro'
    )

    recall_macro = recall_score(
        y_test,
        y_pred,
        average='macro'
    )

    f1_macro = f1_score(
        y_test,
        y_pred,
        average='macro'
    )

    f1_weighted = f1_score(
        y_test,
        y_pred,
        average='weighted'
    )

    print("XGBOOST RESULTS")

    print("Accuracy:", accuracy)
    print("Precision Macro:", precision_macro)
    print("Recall Macro:", recall_macro)
    print("F1 Macro:", f1_macro)
    print("F1 Weighted:", f1_weighted)

    return {
        "Accuracy": accuracy,
        "Macro Precision": precision_macro,
        "Macro Recall": recall_macro,
        "Macro F1-score": f1_macro,
        "Weighted F1-score": f1_weighted
    }


# FUNCTION : CLASSIFICATION REPORT

def display_xgboost_report(y_test, y_pred):
    """
    Generate and display the classification report
    using the predefined AAMI class order.
    """

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1, 2],
            target_names=class_names,
            digits=6,
            zero_division=0
        )
    )


# FUNCTION : CONFUSION MATRIX

def display_xgboost_confusion_matrix(y_test, y_pred):
    """
    Calculate and display the confusion matrix
    using the predefined AAMI class order.
    """

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2]
    )

    print("\nConfusion Matrix:")
    print(cm)

    return cm

In [ ]:

# MODEL EXECUTION

# 1. Train XGBoost
xgb = train_xgboost(X_train, y_train)


# 2. Prediction
y_pred = predict_xgboost(xgb, X_test)


# 3. Evaluation metrics
xgb_results = evaluate_xgboost(
    y_test,
    y_pred
)


# 4. Classification report
display_xgboost_report(
    y_test,
    y_pred
)


# 5. Confusion matrix
cm = display_xgboost_confusion_matrix(
    y_test,
    y_pred
)

# CNN Baseline

In [12]:
# CNN BASELINE
# FUNCTION DEFINITIONS

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# 0. REPRODUCIBILITY

SEED = 42
OUTPUT_DIR = "output working"
CLASS_NAMES = ["N", "S", "V"]


def set_reproducibility(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    print("Random seed:", seed)


def prepare_output_directory(output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)


# 1. LOAD AND PREPARE DATA

def load_data(output_dir=OUTPUT_DIR):
    print("LOADING DATA")

    X_train = np.load(os.path.join(output_dir, "X_train.npy"))
    X_test = np.load(os.path.join(output_dir, "X_test.npy"))
    y_train = np.load(os.path.join(output_dir, "y_train.npy"))
    y_test = np.load(os.path.join(output_dir, "y_test.npy"))

    print("Data loaded successfully")
    print("X_train shape:", X_train.shape)
    print("X_test shape :", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test shape :", y_test.shape)

    return X_train, X_test, y_train, y_test


def reshape_data(X_train, X_test, y_train, y_test):
    X_train = X_train.reshape(X_train.shape[0], 360, 1)
    X_test = X_test.reshape(X_test.shape[0], 360, 1)

    y_train_cat = to_categorical(y_train, 3)
    y_test_cat = to_categorical(y_test, 3)

    print("RESHAPE")
    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)

    return X_train, X_test, y_train_cat, y_test_cat


# 2. CNN MODEL

def build_cnn_model():
    print("BUILDING CNN MODEL")

    model = Sequential()

    model.add(
        Conv1D(
            32,
            kernel_size=5,
            activation="relu",
            input_shape=(360, 1)
        )
    )

    model.add(MaxPooling1D(pool_size=2))

    model.add(
        Conv1D(
            64,
            kernel_size=5,
            activation="relu"
        )
    )

    model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())

    model.add(Dense(128, activation="relu"))

    # Dropout is required for Monte Carlo Dropout
    model.add(Dropout(0.5))

    model.add(Dense(3, activation="softmax"))

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()
    return model


# 3. TRAIN MODEL

def train_cnn_model(model, X_train, y_train_cat):
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    print("TRAINING MODEL")

    history = model.fit(
        X_train,
        y_train_cat,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=1
    )

    return history


# 4. PREDICTIONS AND CLASSIFICATION METRICS

def predict_cnn(model, X_test):
    print("MODEL EVALUATION")

    y_pred_prob = model.predict(
        X_test,
        batch_size=128,
        verbose=1
    )

    y_pred = np.argmax(y_pred_prob, axis=1)

    return y_pred_prob, y_pred


def calculate_cnn_metrics(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)

    precision_macro = precision_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    print("CNN RESULTS")
    print("Accuracy:", round(accuracy, 4))
    print("Precision Macro:", round(precision_macro, 4))
    print("Recall Macro:", round(recall_macro, 4))
    print("F1-score Macro:", round(f1_macro, 4))

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro
    }


def display_classification_report(y_test, y_pred):
    print("CLASSIFICATION REPORT")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1, 2],
            target_names=CLASS_NAMES,
            digits=6,
            zero_division=0
        )
    )


# 5. CONFUSION MATRIX

def calculate_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2]
    )

    print("CONFUSION MATRIX - RAW COUNTS")

    print("Rows = True Class")
    print("Columns = Predicted Class\n")
    print("             Predicted")
    print("             N       S       V")
    print(f"True N   {cm[0,0]:7d} {cm[0,1]:7d} {cm[0,2]:7d}")
    print(f"True S   {cm[1,0]:7d} {cm[1,1]:7d} {cm[1,2]:7d}")
    print(f"True V   {cm[2,0]:7d} {cm[2,1]:7d} {cm[2,2]:7d}")

    return cm


def plot_confusion_matrix(cm, output_dir=OUTPUT_DIR):
    plt.figure(figsize=(7, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar=True,
        linewidths=0.5
    )

    plt.title("Confusion Matrix - CNN")
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.tight_layout()

    plt.savefig(
        os.path.join(output_dir, "CNN_confusion_matrix_raw_counts.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 6. TRAINING CURVE

def plot_training_curve(history, output_dir=OUTPUT_DIR):
    plt.figure(figsize=(7, 5))

    plt.plot(
        history.history["accuracy"],
        label="Train Accuracy"
    )

    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    plt.title("Training and Validation Accuracy - CNN")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        os.path.join(output_dir, "CNN_training_curve.png"),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()




In [ ]:
# CNN BASELINE
# EXECUTION

# 0. Reproducibility and output directory
set_reproducibility()
prepare_output_directory()


# 1. Load data
X_train, X_test, y_train, y_test = load_data()


# 2. Reshape data
X_train, X_test, y_train_cat, y_test_cat = reshape_data(
    X_train,
    X_test,
    y_train,
    y_test
)


# 3. Build CNN model
model = build_cnn_model()


# 4. Train model
history = train_cnn_model(
    model,
    X_train,
    y_train_cat
)


# 5. Normal predictions
y_pred_prob, y_pred = predict_cnn(
    model,
    X_test
)


# 6. Classification metrics
cnn_results = calculate_cnn_metrics(
    y_test,
    y_pred
)


# 7. Classification report
display_classification_report(
    y_test,
    y_pred
)


# 8. Confusion matrix - raw counts
cm = calculate_confusion_matrix(
    y_test,
    y_pred
)

plot_confusion_matrix(cm)


# 9. Training curve
plot_training_curve(history)




# CNN–LSTM

In [15]:
# CNN-LSTM BASELINE
# FUNCTION DEFINITIONS

import numpy as np
import random
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)


# 0. REPRODUCIBILITY

SEED = 42
CLASS_NAMES = ["N", "S", "V"]


def set_reproducibility(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)


# 1. LOAD DATA

def load_data():
    X_train = np.load("path/X_train.npy")
    X_test = np.load("path/X_test.npy")
    y_train = np.load("path/y_train.npy")
    y_test = np.load("path/y_test.npy")

    print("Data loaded successfully")
    print("X_train shape:", X_train.shape)
    print("X_test shape:", X_test.shape)

    return X_train, X_test, y_train, y_test


# 2. RESHAPE DATA

def reshape_data(X_train, X_test, y_train):
    X_train = X_train.reshape(X_train.shape[0], 360, 1)
    X_test = X_test.reshape(X_test.shape[0], 360, 1)

    y_train_cat = to_categorical(y_train, 3)

    return X_train, X_test, y_train_cat


# 3. BUILD CNN-LSTM MODEL

def build_cnn_lstm_model():
    model = Sequential()

    # First convolutional block
    model.add(
        Conv1D(
            filters=32,
            kernel_size=5,
            activation="relu",
            input_shape=(360, 1)
        )
    )

    model.add(MaxPooling1D(pool_size=2))

    # Second convolutional block
    model.add(
        Conv1D(
            filters=64,
            kernel_size=5,
            activation="relu"
        )
    )

    model.add(MaxPooling1D(pool_size=2))

    # LSTM layer
    model.add(
        LSTM(units=64)
    )

    # Classification layers
    model.add(
        Dense(
            128,
            activation="relu"
        )
    )

    model.add(Dropout(0.5))

    model.add(
        Dense(
            3,
            activation="softmax"
        )
    )

    # Compile model
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()

    return model


# 4. TRAIN MODEL

def train_cnn_lstm_model(model, X_train, y_train_cat):
    history = model.fit(
        X_train,
        y_train_cat,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        verbose=1
    )

    return history


# 5. PREDICTION

def predict_cnn_lstm(model, X_test):
    y_pred_prob = model.predict(
        X_test,
        batch_size=128,
        verbose=1
    )

    y_pred = np.argmax(y_pred_prob, axis=1)

    return y_pred_prob, y_pred


# 6. CLASSIFICATION REPORT

def display_classification_report(y_test, y_pred):
    print("CNN-LSTM CLASSIFICATION REPORT")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1, 2],
            target_names=CLASS_NAMES,
            digits=6,
            zero_division=0
        )
    )


# 7. CONFUSION MATRIX

def calculate_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2]
    )

    print("CNN-LSTM CONFUSION MATRIX")

    print("Rows = True Class")
    print("Columns = Predicted Class")
    print("Class order: N, S, V\n")
    print(cm)

    return cm

In [ ]:
# CNN-LSTM BASELINE
# EXECUTION

# 0. Reproducibility
set_reproducibility()


# 1. Load data
X_train, X_test, y_train, y_test = load_data()


# 2. Reshape data
X_train, X_test, y_train_cat = reshape_data(
    X_train,
    X_test,
    y_train
)


# 3. Build CNN-LSTM model
model = build_cnn_lstm_model()


# 4. Train model
history = train_cnn_lstm_model(
    model,
    X_train,
    y_train_cat
)


# 5. Prediction
y_pred_prob, y_pred = predict_cnn_lstm(
    model,
    X_test
)


# 6. Classification report
display_classification_report(
    y_test,
    y_pred
)


# 7. Confusion matrix - raw counts
cm = calculate_confusion_matrix(
    y_test,
    y_pred
)

# MS-CNN–BiLSTM–Attention 

In [12]:
# MS-CNN-BiLSTM-ATTENTION + FULL EVALUATION + MC DROPOUT
# FUNCTIONS 



import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, BatchNormalization,
    Bidirectional, LSTM, Dense, Dropout,
    GlobalAveragePooling1D, concatenate, Softmax
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score, precision_recall_curve, average_precision_score
)


# EXPERIMENT CONFIGURATION

SEED = 42
DATA_DIR = "path working"

EPOCHS = 10
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.10
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.5

CLASS_NAMES = ["N", "S", "V"]
CLASS_IDS = [0, 1, 2]

# Monte Carlo Dropout configuration
MC_T = 30
MC_BATCH_SIZE = 256


# REPRODUCIBILITY

def set_reproducibility(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    print("=" * 70)
    print("REPRODUCIBILITY")
    print("=" * 70)
    print("Random seed:", seed)
    print("TensorFlow version:", tf.__version__)


# LOAD AND PREPARE DATA

def load_and_prepare_data(data_dir=DATA_DIR):

    X_train = np.load(os.path.join(data_dir, "X_train.npy"))
    X_test = np.load(os.path.join(data_dir, "X_test.npy"))
    y_train = np.load(os.path.join(data_dir, "y_train.npy"))
    y_test = np.load(os.path.join(data_dir, "y_test.npy"))

    print("\n" + "=" * 70)
    print("DATASET")
    print("=" * 70)
    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)
    print("X_test :", X_test.shape)
    print("y_test :", y_test.shape)
    print("Total samples:", len(X_train) + len(X_test))

    print("\n" + "=" * 70)
    print("CLASS DISTRIBUTION")
    print("=" * 70)

    for split_name, labels in [
        ("Training", y_train),
        ("Test", y_test)
    ]:
        print(f"\n{split_name} set:")

        for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):
            count = int(np.sum(labels == class_id))
            percentage = count / len(labels) * 100
            print(f"{class_name}: {count:,} ({percentage:.2f}%)")

    X_train = X_train.reshape(X_train.shape[0], 360, 1)
    X_test = X_test.reshape(X_test.shape[0], 360, 1)

    print("\n" + "=" * 70)
    print("INPUT SHAPE")
    print("=" * 70)
    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)

    y_train_cat = to_categorical(y_train, num_classes=3)
    y_test_cat = to_categorical(y_test, num_classes=3)

    return (
        X_train, X_test, y_train, y_test,
        y_train_cat, y_test_cat
    )


# BUILD PROPOSED MODEL

def build_model():

    input_layer = Input(shape=(360, 1), name="ECG_Input")

    # Multi-scale CNN branches
    conv3 = Conv1D(
        filters=32,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="Conv1D_Kernel_3"
    )(input_layer)

    conv5 = Conv1D(
        filters=32,
        kernel_size=5,
        padding="same",
        activation="relu",
        name="Conv1D_Kernel_5"
    )(input_layer)

    conv7 = Conv1D(
        filters=32,
        kernel_size=7,
        padding="same",
        activation="relu",
        name="Conv1D_Kernel_7"
    )(input_layer)

    merged = concatenate(
        [conv3, conv5, conv7],
        axis=-1,
        name="MultiScale_Concatenation"
    )

    # Feature refinement
    x = Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="Conv1D_Feature_Refinement"
    )(merged)

    x = BatchNormalization(name="Batch_Normalization")(x)
    x = MaxPooling1D(pool_size=2, name="MaxPooling")(x)

    # Bidirectional LSTM
    x = Bidirectional(
        LSTM(64, return_sequences=True),
        name="BiLSTM"
    )(x)

    # Temporal attention
    attention_scores = Dense(
        1,
        activation="tanh",
        name="Attention_Scores"
    )(x)

    attention_weights = Softmax(
        axis=1,
        name="Temporal_Attention_Weights"
    )(attention_scores)

    x = x * attention_weights

    # Global average pooling and classifier
    x = GlobalAveragePooling1D(
        name="Global_Average_Pooling"
    )(x)

    x = Dense(
        128,
        activation="relu",
        name="Dense_128"
    )(x)

    x = Dropout(
        DROPOUT_RATE,
        name="Dropout_0.5"
    )(x)

    output_layer = Dense(
        3,
        activation="softmax",
        name="Output_3_Classes"
    )(x)

    model = Model(
        inputs=input_layer,
        outputs=output_layer,
        name="MS_CNN_BiLSTM_Attention"
    )

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# TRAIN MODEL

def train_model(model, X_train, y_train_cat):

    print("\n" + "=" * 70)
    print("TRAINING")
    print("=" * 70)
    print("Epochs:", EPOCHS)
    print("Batch size:", BATCH_SIZE)
    print("Validation split:", VALIDATION_SPLIT)
    print("Optimizer: Adam")
    print("Learning rate:", LEARNING_RATE)
    print("Dropout:", DROPOUT_RATE)
    print("Class weighting: None")
    print("Oversampling: None")

    history = model.fit(
        X_train,
        y_train_cat,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1
    )

    model.save("MS_CNN_BiLSTM_Attention_full.keras")
    print("Model saved: MS_CNN_BiLSTM_Attention_full.keras")

    return history


# PLOT TRAINING HISTORY

def plot_training_history(history):

    print("\n" + "=" * 70)
    print("FINAL TRAINING / VALIDATION RESULTS")
    print("=" * 70)

    for metric, label in [
        ("accuracy", "accuracy"),
        ("val_accuracy", "validation accuracy"),
        ("loss", "loss"),
        ("val_loss", "validation loss")
    ]:
        print(f"Final {label}: {history.history[metric][-1]:.4f}")

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Proposed Model - Accuracy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_accuracy_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Proposed Model - Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_loss_curve.png", dpi=300, bbox_inches="tight")
    plt.show()


# COMPLETE MODEL EVALUATION

def evaluate_model(y_test, y_test_cat, y_pred_prob):

    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)

    precision_macro = precision_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    recall_macro = recall_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    f1_macro = f1_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    f1_weighted = f1_score(
        y_test, y_pred,
        average="weighted",
        zero_division=0
    )

    print("\n" + "=" * 70)
    print("OVERALL TEST RESULTS")
    print("=" * 70)
    print(f"Accuracy          : {accuracy:.4f}")
    print(f"Macro Precision   : {precision_macro:.4f}")
    print(f"Macro Recall      : {recall_macro:.4f}")
    print(f"Macro F1-score    : {f1_macro:.4f}")
    print(f"Weighted F1-score : {f1_weighted:.4f}")

    # Classification report
    report = classification_report(
        y_test,
        y_pred,
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )

    print("\n" + "=" * 70)
    print("CLASSIFICATION REPORT")
    print("=" * 70)
    print(report)

    with open(
        "classification_report_full.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(report)

    # Confusion matrix
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=CLASS_IDS
    )

    cm_df = pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    )

    print("\n" + "=" * 70)
    print("CONFUSION MATRIX - RAW COUNTS")
    print("=" * 70)
    print("Rows = True class | Columns = Predicted class")
    print(cm_df)

    print("\nRow totals:")
    for i, name in enumerate(CLASS_NAMES):
        print(f"{name}: {cm[i].sum():,}")

    print("Total test samples:", cm.sum())

    cm_df.to_csv("confusion_matrix_raw.csv")

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        linewidths=0.5,
        cbar=True
    )
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title("Proposed Model Confusion Matrix")
    plt.tight_layout()
    plt.savefig("confusion_matrix_raw.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Normalized confusion matrix
    cm_normalized = cm.astype(np.float64) / cm.sum(
        axis=1,
        keepdims=True
    )

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        vmin=0,
        vmax=1,
        cbar=True
    )
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title("Normalized Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        "confusion_matrix_normalized.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    # One-vs-rest TP / FP / FN / TN
    total_test = cm.sum()
    error_rows = []

    for i, class_name in enumerate(CLASS_NAMES):

        tp = int(cm[i, i])
        fn = int(cm[i, :].sum() - tp)
        fp = int(cm[:, i].sum() - tp)
        tn = int(total_test - tp - fn - fp)

        class_precision = tp / (tp + fp) if tp + fp else 0.0
        class_recall = tp / (tp + fn) if tp + fn else 0.0

        class_f1 = (
            2 * class_precision * class_recall /
            (class_precision + class_recall)
            if class_precision + class_recall else 0.0
        )

        error_rows.append({
            "Class": class_name,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "Precision": class_precision,
            "Recall": class_recall,
            "F1": class_f1
        })

        print(
            f"\nClass {class_name}: TP={tp}, FP={fp}, FN={fn}, TN={tn}, "
            f"Precision={class_precision:.4f}, Recall={class_recall:.4f}, "
            f"F1={class_f1:.4f}"
        )

    error_df = pd.DataFrame(error_rows)
    error_df.to_csv("one_vs_rest_error_analysis.csv", index=False)

    print("\nOne-vs-rest error analysis:")
    print(error_df.to_string(index=False))

    # ROC curves and AUC
    roc_auc_values = {}

    plt.figure(figsize=(8, 6))

    for i, class_name in enumerate(CLASS_NAMES):

        y_true_binary = (y_test == i).astype(int)

        fpr, tpr, _ = roc_curve(
            y_true_binary,
            y_pred_prob[:, i]
        )

        roc_auc = auc(fpr, tpr)
        roc_auc_values[class_name] = roc_auc

        print(f"{class_name} ROC-AUC: {roc_auc:.4f}")

        plt.plot(
            fpr,
            tpr,
            label=f"{class_name} (AUC = {roc_auc:.4f})"
        )

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves - Proposed Model")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("ROC_curves.png", dpi=300, bbox_inches="tight")
    plt.show()

    macro_roc_auc = roc_auc_score(
        y_test_cat,
        y_pred_prob,
        average="macro"
    )

    weighted_roc_auc = roc_auc_score(
        y_test_cat,
        y_pred_prob,
        average="weighted"
    )

    print(f"Macro ROC-AUC    : {macro_roc_auc:.4f}")
    print(f"Weighted ROC-AUC : {weighted_roc_auc:.4f}")

    # Precision-recall curves and AP
    auprc_values = {}

    plt.figure(figsize=(8, 6))

    for i, class_name in enumerate(CLASS_NAMES):

        y_true_binary = (y_test == i).astype(int)

        precision_curve, recall_curve, _ = precision_recall_curve(
            y_true_binary,
            y_pred_prob[:, i]
        )

        ap = average_precision_score(
            y_true_binary,
            y_pred_prob[:, i]
        )

        auprc_values[class_name] = ap

        print(f"{class_name} AUPRC: {ap:.4f}")

        plt.plot(
            recall_curve,
            precision_curve,
            label=f"{class_name} (AP = {ap:.4f})"
        )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curves - Proposed Model")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("PR_curves.png", dpi=300, bbox_inches="tight")
    plt.show()

    macro_auprc = float(np.mean(list(auprc_values.values())))

    # Prediction confidence
    max_prob = np.max(y_pred_prob, axis=1)

    correct_mask = y_pred == y_test
    incorrect_mask = y_pred != y_test

    mean_correct_conf = (
        float(max_prob[correct_mask].mean())
        if np.any(correct_mask) else np.nan
    )

    mean_incorrect_conf = (
        float(max_prob[incorrect_mask].mean())
        if np.any(incorrect_mask) else np.nan
    )

    print("\n" + "=" * 70)
    print("PREDICTION CONFIDENCE")
    print("=" * 70)
    print(f"Mean confidence - correct predictions   : {mean_correct_conf:.4f}")
    print(f"Mean confidence - incorrect predictions : {mean_incorrect_conf:.4f}")
    print("Correct predictions   :", int(correct_mask.sum()))
    print("Incorrect predictions :", int(incorrect_mask.sum()))

    # Confidence by true class
    print("\n" + "=" * 70)
    print("CONFIDENCE BY TRUE CLASS")
    print("=" * 70)

    confidence_by_class = []

    for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):

        mask = y_test == class_id
        mean_conf = float(max_prob[mask].mean())

        confidence_by_class.append({
            "Class": class_name,
            "Mean_Confidence": mean_conf
        })

        print(f"{class_name}: mean confidence = {mean_conf:.4f}")

    pd.DataFrame(confidence_by_class).to_csv(
        "confidence_by_true_class.csv",
        index=False
    )

    # Misclassification counts
    print("\n" + "=" * 70)
    print("MISCLASSIFICATION ANALYSIS")
    print("=" * 70)

    misclassification_rows = []

    for true_id, true_name in zip(CLASS_IDS, CLASS_NAMES):

        print(f"\nTrue class: {true_name}")

        for pred_id, pred_name in zip(CLASS_IDS, CLASS_NAMES):

            if true_id != pred_id:

                n_errors = int(np.sum(
                    (y_test == true_id) & (y_pred == pred_id)
                ))

                if n_errors:
                    print(f"  {true_name} -> {pred_name}: {n_errors}")

                misclassification_rows.append({
                    "True_Class": true_name,
                    "Predicted_Class": pred_name,
                    "Count": n_errors
                })

    pd.DataFrame(misclassification_rows).to_csv(
        "misclassification_analysis.csv",
        index=False
    )

    # Final results summary
    summary_df = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1",
            "Weighted F1",
            "Macro ROC-AUC",
            "Weighted ROC-AUC",
            "Macro AUPRC"
        ],
        "Value": [
            accuracy,
            precision_macro,
            recall_macro,
            f1_macro,
            f1_weighted,
            macro_roc_auc,
            weighted_roc_auc,
            macro_auprc
        ]
    })

    print("\n" + "=" * 70)
    print("FINAL RESULTS SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))

    summary_df.to_csv("final_results_summary.csv", index=False)

    # Export predictions
    predictions_df = pd.DataFrame({
        "true_label": [CLASS_NAMES[i] for i in y_test],
        "predicted_label": [CLASS_NAMES[i] for i in y_pred],
        "prob_N": y_pred_prob[:, 0],
        "prob_S": y_pred_prob[:, 1],
        "prob_V": y_pred_prob[:, 2],
        "max_confidence": max_prob,
        "correct": correct_mask
    })

    predictions_df.to_csv("test_predictions.csv", index=False)

    return {
        "y_pred": y_pred,
        "y_pred_prob": y_pred_prob,
        "cm": cm,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "roc_auc_values": roc_auc_values,
        "macro_roc_auc": macro_roc_auc,
        "weighted_roc_auc": weighted_roc_auc,
        "auprc_values": auprc_values,
        "macro_auprc": macro_auprc,
        "error_df": error_df,
        "summary_df": summary_df,
        "max_prob": max_prob,
        "correct_mask": correct_mask,
        "incorrect_mask": incorrect_mask
    }


# MONTE CARLO DROPOUT

def monte_carlo_dropout(
    model,
    X_test,
    T=MC_T,
    batch_size=MC_BATCH_SIZE
):

    """Repeated stochastic forward passes with Dropout active."""

    mc_mean_batches = []
    mc_variance_batches = []

    for start in range(0, len(X_test), batch_size):

        batch = X_test[start:start + batch_size]

        batch_predictions = np.stack(
            [
                model(batch, training=True).numpy()
                for _ in range(T)
            ],
            axis=0
        )

        mc_mean_batches.append(
            np.mean(batch_predictions, axis=0)
        )

        mc_variance_batches.append(
            np.var(batch_predictions, axis=0)
        )

    mc_mean_prob = np.concatenate(mc_mean_batches, axis=0)
    mc_variance = np.concatenate(mc_variance_batches, axis=0)

    return mc_mean_prob, mc_variance


# TRUE-CLASS UNCERTAINTY ANALYSIS

def analyze_mc_uncertainty(
    y_test,
    mc_mean_prob,
    mc_variance
):

    # Mean class-probability variance for each test example
    sample_uncertainty = np.mean(mc_variance, axis=1)

    uncertainty_df = pd.DataFrame({
        "true_label": [CLASS_NAMES[i] for i in y_test],
        "mc_mean_prob_N": mc_mean_prob[:, 0],
        "mc_mean_prob_S": mc_mean_prob[:, 1],
        "mc_mean_prob_V": mc_mean_prob[:, 2],
        "variance_N": mc_variance[:, 0],
        "variance_S": mc_variance[:, 1],
        "variance_V": mc_variance[:, 2],
        "mean_predictive_variance": sample_uncertainty
    })

    uncertainty_df.to_csv(
        "MC_dropout_uncertainty.csv",
        index=False
    )

    print("\n" + "=" * 70)
    print("MONTE CARLO DROPOUT UNCERTAINTY")
    print("=" * 70)
    print("Number of MC passes:", MC_T)
    print("Mean uncertainty:", float(np.mean(sample_uncertainty)))
    print("Maximum uncertainty:", float(np.max(sample_uncertainty)))
    print("Minimum uncertainty:", float(np.min(sample_uncertainty)))

    class_means = []
    class_stds = []
    class_counts = []

    for class_id, class_name in zip(CLASS_IDS, CLASS_NAMES):

        values = sample_uncertainty[y_test == class_id]

        mean_value = float(np.mean(values))
        std_value = float(np.std(values))

        class_means.append(mean_value)
        class_stds.append(std_value)
        class_counts.append(len(values))

        print(
            f"Class {class_name}: mean={mean_value:.8f}, "
            f"std={std_value:.8f}, n={len(values)}"
        )

    uncertainty_summary = pd.DataFrame({
        "Class": CLASS_NAMES,
        "Mean_Uncertainty": class_means,
        "Std_Uncertainty": class_stds,
        "Count": class_counts
    })

    uncertainty_summary.to_csv(
        "MC_uncertainty_by_true_class.csv",
        index=False
    )

    # Bar plot
    plt.figure(figsize=(8, 6))
    plt.bar(
        CLASS_NAMES,
        class_means,
        yerr=class_stds,
        capsize=5
    )
    plt.xlabel("True Class")
    plt.ylabel("Mean Predictive Variance")
    plt.title("MC Dropout Uncertainty by True Class")
    plt.tight_layout()
    plt.savefig(
        "MC_uncertainty_by_true_class.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    # Boxplot
    plt.figure(figsize=(8, 6))
    sns.boxplot(
        x=uncertainty_df["true_label"],
        y=uncertainty_df["mean_predictive_variance"],
        order=CLASS_NAMES
    )
    plt.xlabel("True Class")
    plt.ylabel("Mean Predictive Variance")
    plt.title("MC Dropout Uncertainty Distribution by True Class")
    plt.tight_layout()
    plt.savefig(
        "MC_uncertainty_boxplot.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    return uncertainty_df, uncertainty_summary
# FINAL SCIENTIFIC INTERPRETATION

def display_final_interpretation(uncertainty_statistics):

    print("\n" + "=" * 70)
    print("FINAL SCIENTIFIC INTERPRETATION")
    print("=" * 70)

    print(
        "The Monte Carlo Dropout analysis was performed to estimate "
        "the predictive uncertainty of the proposed model."
    )

    print(
        "\nThe mean predictive variance was calculated for each "
        "heartbeat and then analyzed according to its true class."
    )

    print("\nUncertainty statistics by true class:")

    for _, row in uncertainty_statistics.iterrows():

        class_name = row["Class"]
        mean_uncertainty = row["Mean_Uncertainty"]
        std_uncertainty = row["Std_Uncertainty"]

        print(
            f"\nClass {class_name}: "
            f"Mean uncertainty = {mean_uncertainty:.8f}, "
            f"Standard deviation = {std_uncertainty:.8f}"
        )

    print("\n" + "-" * 70)
    print("INTERPRETATION")
    print("-" * 70)

    means = uncertainty_statistics.set_index("Class")[
        "Mean_Uncertainty"
    ]

    lowest_class = means.idxmin()
    highest_class = means.idxmax()

    print(
        f"The {lowest_class} class presents the lowest mean predictive "
        "variance among the evaluated classes."
    )

    print(
        f"The {highest_class} class presents the highest mean predictive "
        "variance among the evaluated classes."
    )

    print(
        "\nThese results describe the uncertainty estimated by MC Dropout. "
        "They should be interpreted alongside the classification metrics "
        "and class-wise error analysis."
    )

    print("=" * 70)

In [ ]:
# EXECUTION  MS-CNN-BiLSTM-ATTENTION + FULL EVALUATION + MC DROPOUT

# 1. Reproducibility
set_reproducibility(SEED)


# 2. Load and prepare the existing train/test arrays
(
    X_train, X_test, y_train, y_test,
    y_train_cat, y_test_cat
) = load_and_prepare_data(DATA_DIR)


# 3. Build the proposed architecture
print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

model = build_model()
model.summary()

print("\nTotal parameters:", model.count_params())


# 4. Train the model
history = train_model(
    model,
    X_train,
    y_train_cat
)


# 5. Training history and curves
plot_training_history(history)


# 6. Standard test predictions
print("\n" + "=" * 70)
print("PREDICTIONS")
print("=" * 70)

y_pred_prob = model.predict(
    X_test,
    batch_size=256,
    verbose=1
)

print("Prediction probability shape:", y_pred_prob.shape)


# 7. Complete evaluation
# Metrics, classification report, confusion matrices,
# ROC/PR curves, confidence, FP/FN, and misclassifications

results = evaluate_model(
    y_test,
    y_test_cat,
    y_pred_prob
)


# 8. Monte Carlo Dropout analysis

print("\nRunning Monte Carlo Dropout...")

mc_mean_prob, mc_variance = monte_carlo_dropout(
    model,
    X_test,
    T=MC_T,
    batch_size=MC_BATCH_SIZE
)


# 9. True-class uncertainty analysis
uncertainty_df, uncertainty_summary = analyze_mc_uncertainty(
    y_test,
    mc_mean_prob,
    mc_variance
)


# 10. Final summary
print("\n" + "=" * 70)
print("EXECUTION COMPLETED")
print("=" * 70)

print(
    "Standard evaluation and Monte Carlo uncertainty outputs "
    "have been saved."
)

print("Available variables for subsequent analysis:")
print("""
y_test, y_pred, y_pred_prob, cm, history, results,
mc_mean_prob, mc_variance, uncertainty_df, uncertainty_summary
""")

# 11. Final scientific interpretation
display_final_interpretation(
    uncertainty_summary
)

# Ablation  & STATISTICAL ANALYSIS

## Ablation Without BiLSTM

In [ ]:
# PRESERVE FULL MODEL PREDICTIONS

import numpy as np

# Check that the required variables exist
assert "y_test" in globals(), "y_test is not found."
assert "results" in globals(), "results is not found."
assert "y_pred" in results, "results does not contain the key 'y_pred'."

# Copy the labels and predictions from the full model
y_test_full = np.asarray(y_test).copy()
y_pred_full = np.asarray(results["y_pred"]).copy()

# Identify correct and incorrect predictions of the full model
correctness_full = (
    y_pred_full == y_test_full
).astype(np.uint8)

print("=" * 70)
print("FULL MODEL PREDICTIONS PRESERVED")
print("=" * 70)

print("y_test_full shape:", y_test_full.shape)
print("y_pred_full shape:", y_pred_full.shape)
print("Correct predictions:", correctness_full.sum())
print("Incorrect predictions:", len(correctness_full) - correctness_full.sum())
print("Full model accuracy:", correctness_full.mean())

assert len(y_test_full) == len(y_pred_full)

In [ ]:
# ABLATION - WITHOUT BiLSTM
# MS-CNN + TEMPORAL ATTENTION

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    Concatenate,
    Multiply,
    Softmax
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)

from sklearn.preprocessing import label_binarize


# 1. REPRODUCIBILITY

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 70)
print("ABLATION - WITHOUT BiLSTM")
print("MS-CNN + TEMPORAL ATTENTION")
print("=" * 70)

print("Random seed:", SEED)


# 2. LOAD SAME TRAIN / TEST DATA

DATA_DIR = "path working"

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Total samples:", len(X_train) + len(X_test))


# 3. VERIFY TEST SET AGAINST FULL MODEL

assert "y_test_full" in globals(), (
    "Cell preserve all ful model predictions has not been executed. "
    "Please preserve the full-model predictions first."
)

assert np.array_equal(y_test_full, y_test), (
    "WARNING: Test labels do not match "
    "the labels saved for the full model."
)

assert len(y_pred_full) == len(y_test), (
    "The number of full-model predictions differs "
    "from the number of test labels."
)

print("\nTest labels match the saved full-model labels.")


# 4. CLASS DISTRIBUTION

class_names = ["N", "S", "V"]
class_ids = [0, 1, 2]
num_classes = 3

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print("\nTraining:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_train == class_id)
    percentage = count / len(y_train) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")

print("\nTest:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_test == class_id)
    percentage = count / len(y_test) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")


# 5. RESHAPE

X_train = X_train.reshape(X_train.shape[0], 360, 1)
X_test = X_test.reshape(X_test.shape[0], 360, 1)

print("\n" + "=" * 70)
print("INPUT SHAPE")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)




inputs = Input(
    shape=(360, 1),
    name="ECG_Input"
)


# 7. MULTI-SCALE CNN

branch_3 = Conv1D(
    filters=32,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Conv1D_K3"
)(inputs)

branch_5 = Conv1D(
    filters=32,
    kernel_size=5,
    padding="same",
    activation="relu",
    name="Conv1D_K5"
)(inputs)

branch_7 = Conv1D(
    filters=32,
    kernel_size=7,
    padding="same",
    activation="relu",
    name="Conv1D_K7"
)(inputs)


# 8. CONCATENATION

x = Concatenate(
    axis=-1,
    name="MultiScale_Concatenation"
)([branch_3, branch_5, branch_7])


# 9. FEATURE REFINEMENT

x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Feature_Refinement_Conv"
)(x)

x = BatchNormalization(
    name="BatchNormalization"
)(x)

x = MaxPooling1D(
    pool_size=2,
    name="MaxPooling"
)(x)


# 10. ATTENTION

attention_scores = Dense(
    1,
    activation="tanh",
    name="Attention_Score"
)(x)

attention_weights = Softmax(
    axis=1,
    name="Temporal_Attention_Weights"
)(attention_scores)

x = Multiply(
    name="Attention_Multiply"
)([x, attention_weights])


# 11. GLOBAL AVERAGE POOLING

x = GlobalAveragePooling1D(
    name="GlobalAveragePooling"
)(x)


# 12. CLASSIFIER

x = Dense(
    128,
    activation="relu",
    name="Dense_128"
)(x)

x = Dropout(
    0.5,
    name="Dropout_0.5"
)(x)

outputs = Dense(
    3,
    activation="softmax",
    name="Output_3_Classes"
)(x)


# 13. CREATE MODEL

model_no_bilstm = Model(
    inputs=inputs,
    outputs=outputs,
    name="MS_CNN_Attention_Without_BiLSTM"
)


# 14. ARCHITECTURE SUMMARY

print("\n" + "=" * 70)
print("WITHOUT BiLSTM ARCHITECTURE")
print("=" * 70)

model_no_bilstm.summary()

print("\nTotal parameters:", model_no_bilstm.count_params())


# 15. COMPILE

model_no_bilstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 16. TRAINING CONFIGURATION

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Epochs: 10")
print("Batch size: 128")
print("Validation split: 10%")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Dropout: 0.5")
print("Class weighting: None")
print("Oversampling: None")
print("Data augmentation: None")
print("BiLSTM: REMOVED")
print("Attention: RETAINED")


# 17. TRAIN

history_no_bilstm = model_no_bilstm.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.10,
    shuffle=True,
    verbose=1
)


# 18. TRAINING RESULTS

print("\n" + "=" * 70)
print("FINAL TRAINING / VALIDATION RESULTS")
print("=" * 70)

print(
    f"Final training accuracy: "
    f"{history_no_bilstm.history['accuracy'][-1]:.4f}"
)

print(
    f"Final validation accuracy: "
    f"{history_no_bilstm.history['val_accuracy'][-1]:.4f}"
)

print(
    f"Final training loss: "
    f"{history_no_bilstm.history['loss'][-1]:.4f}"
)

print(
    f"Final validation loss: "
    f"{history_no_bilstm.history['val_loss'][-1]:.4f}"
)


# 19. TRAINING CURVES

plt.figure(figsize=(8, 5))

plt.plot(
    history_no_bilstm.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_no_bilstm.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Without BiLSTM - Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history_no_bilstm.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_no_bilstm.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Without BiLSTM - Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 20. PREDICTIONS

print("\n" + "=" * 70)
print("PREDICTIONS")
print("=" * 70)

y_pred_prob_no_bilstm = model_no_bilstm.predict(
    X_test,
    batch_size=128,
    verbose=1
)

y_pred_no_bilstm = np.argmax(
    y_pred_prob_no_bilstm,
    axis=1
)

print("Prediction shape:", y_pred_no_bilstm.shape)
print("Probability shape:", y_pred_prob_no_bilstm.shape)


# 21. OVERALL METRICS

accuracy_no_bilstm = accuracy_score(
    y_test, y_pred_no_bilstm
)

precision_macro_no_bilstm = precision_score(
    y_test,
    y_pred_no_bilstm,
    average="macro",
    zero_division=0
)

recall_macro_no_bilstm = recall_score(
    y_test,
    y_pred_no_bilstm,
    average="macro",
    zero_division=0
)

f1_macro_no_bilstm = f1_score(
    y_test,
    y_pred_no_bilstm,
    average="macro",
    zero_division=0
)

f1_weighted_no_bilstm = f1_score(
    y_test,
    y_pred_no_bilstm,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("WITHOUT BiLSTM TEST RESULTS")
print("=" * 70)

print(f"Accuracy          : {accuracy_no_bilstm:.4f}")
print(f"Macro Precision   : {precision_macro_no_bilstm:.4f}")
print(f"Macro Recall      : {recall_macro_no_bilstm:.4f}")
print(f"Macro F1-score    : {f1_macro_no_bilstm:.4f}")
print(f"Weighted F1-score : {f1_weighted_no_bilstm:.4f}")


# 22. CLASSIFICATION REPORT

print("\n" + "=" * 70)
print("WITHOUT BiLSTM CLASSIFICATION REPORT")
print("=" * 70)

report_no_bilstm = classification_report(
    y_test,
    y_pred_no_bilstm,
    labels=class_ids,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report_no_bilstm)


# 23. CONFUSION MATRIX - RAW COUNTS

cm_no_bilstm = confusion_matrix(
    y_test,
    y_pred_no_bilstm,
    labels=class_ids
)

print("\n" + "=" * 70)
print("WITHOUT BiLSTM CONFUSION MATRIX - RAW COUNTS")
print("=" * 70)

print("\nRows = True Class")
print("Columns = Predicted Class\n")

cm_no_bilstm_df = pd.DataFrame(
    cm_no_bilstm,
    index=class_names,
    columns=class_names
)

print(cm_no_bilstm_df)
print("\nTotal test samples:", cm_no_bilstm.sum())


# 24. TP / FP / FN / TN

print("\n" + "=" * 70)
print("WITHOUT BiLSTM ONE-VS-REST ERROR ANALYSIS")
print("=" * 70)

total_test_no_bilstm = cm_no_bilstm.sum()
error_table_no_bilstm = []

for i, class_name in enumerate(class_names):

    TP = cm_no_bilstm[i, i]
    FN = cm_no_bilstm[i, :].sum() - TP
    FP = cm_no_bilstm[:, i].sum() - TP
    TN = total_test_no_bilstm - TP - FN - FP

    precision_class = (
        TP / (TP + FP) if (TP + FP) > 0 else 0
    )

    recall_class = (
        TP / (TP + FN) if (TP + FN) > 0 else 0
    )

    f1_class = (
        2 * precision_class * recall_class
        / (precision_class + recall_class)
        if (precision_class + recall_class) > 0
        else 0
    )

    error_table_no_bilstm.append([
        class_name,
        TP,
        FP,
        FN,
        TN,
        precision_class,
        recall_class,
        f1_class
    ])

    print(f"\nClass {class_name}")
    print("TP:", TP)
    print("FP:", FP)
    print("FN:", FN)
    print("TN:", TN)
    print(f"Precision: {precision_class:.4f}")
    print(f"Recall   : {recall_class:.4f}")
    print(f"F1-score : {f1_class:.4f}")


error_df_no_bilstm = pd.DataFrame(
    error_table_no_bilstm,
    columns=[
        "Class", "TP", "FP", "FN", "TN",
        "Precision", "Recall", "F1"
    ]
)

print("\n", error_df_no_bilstm.to_string(index=False))


# 25. CONFUSION MATRIX FIGURE

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm_no_bilstm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    cbar=True
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Without BiLSTM - Confusion Matrix")
plt.tight_layout()
plt.show()


# 26. ROC-AUC

print("\n" + "=" * 70)
print("WITHOUT BiLSTM ROC-AUC ANALYSIS")
print("=" * 70)

y_test_cat_no_bilstm = tf.keras.utils.to_categorical(
    y_test,
    num_classes=3
)

roc_auc_values_no_bilstm = {}

plt.figure(figsize=(8, 6))

for i, class_name in zip(class_ids, class_names):

    y_true_binary = (y_test == i).astype(int)

    fpr, tpr, _ = roc_curve(
        y_true_binary,
        y_pred_prob_no_bilstm[:, i]
    )

    roc_auc = auc(fpr, tpr)
    roc_auc_values_no_bilstm[class_name] = roc_auc

    print(f"{class_name} ROC-AUC: {roc_auc:.4f}")

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {roc_auc:.4f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Without BiLSTM - ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_roc_auc_no_bilstm = roc_auc_score(
    y_test_cat_no_bilstm,
    y_pred_prob_no_bilstm,
    average="macro"
)

weighted_roc_auc_no_bilstm = roc_auc_score(
    y_test_cat_no_bilstm,
    y_pred_prob_no_bilstm,
    average="weighted"
)

print(f"\nMacro ROC-AUC    : {macro_roc_auc_no_bilstm:.4f}")
print(f"Weighted ROC-AUC : {weighted_roc_auc_no_bilstm:.4f}")


# 27. AUPRC

print("\n" + "=" * 70)
print("WITHOUT BiLSTM PRECISION-RECALL / AUPRC")
print("=" * 70)

auprc_values_no_bilstm = {}

plt.figure(figsize=(8, 6))

for i, class_name in zip(class_ids, class_names):

    y_true_binary = (y_test == i).astype(int)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_true_binary,
        y_pred_prob_no_bilstm[:, i]
    )

    ap = average_precision_score(
        y_true_binary,
        y_pred_prob_no_bilstm[:, i]
    )

    auprc_values_no_bilstm[class_name] = ap

    print(f"{class_name} AUPRC: {ap:.4f}")

    plt.plot(
        recall_curve,
        precision_curve,
        label=f"{class_name} (AP = {ap:.4f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Without BiLSTM - Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_auprc_no_bilstm = np.mean(
    list(auprc_values_no_bilstm.values())
)

print(f"\nMacro AUPRC: {macro_auprc_no_bilstm:.4f}")


# 28. PREDICTION CONFIDENCE

max_prob_no_bilstm = np.max(
    y_pred_prob_no_bilstm,
    axis=1
)

correct_mask_no_bilstm = (
    y_pred_no_bilstm == y_test
)

incorrect_mask_no_bilstm = (
    y_pred_no_bilstm != y_test
)

print("\n" + "=" * 70)
print("WITHOUT BiLSTM PREDICTION CONFIDENCE")
print("=" * 70)

print(
    f"Mean confidence - correct: "
    f"{max_prob_no_bilstm[correct_mask_no_bilstm].mean():.4f}"
)

if incorrect_mask_no_bilstm.sum() > 0:
    print(
        f"Mean confidence - incorrect: "
        f"{max_prob_no_bilstm[incorrect_mask_no_bilstm].mean():.4f}"
    )
else:
    print("Mean confidence - incorrect: N/A (no incorrect predictions)")

print(f"Correct predictions: {correct_mask_no_bilstm.sum():,}")
print(f"Incorrect predictions: {incorrect_mask_no_bilstm.sum():,}")


# 29. CONFIDENCE BY CLASS

print("\n" + "=" * 70)
print("WITHOUT BiLSTM CONFIDENCE BY TRUE CLASS")
print("=" * 70)

for i, class_name in enumerate(class_names):

    class_mask = (y_test == i)

    print(
        f"{class_name}: "
        f"{max_prob_no_bilstm[class_mask].mean():.4f}"
    )


# 30. MISCLASSIFICATION ANALYSIS

print("\n" + "=" * 70)
print("WITHOUT BiLSTM MISCLASSIFICATION ANALYSIS")
print("=" * 70)

for true_id, true_name in zip(class_ids, class_names):

    print(f"\nTrue class: {true_name}")

    mask_true = (y_test == true_id)

    for pred_id, pred_name in zip(class_ids, class_names):

        if true_id != pred_id:

            n_errors = np.sum(
                mask_true & (y_pred_no_bilstm == pred_id)
            )

            if n_errors > 0:
                print(f"  {true_name} -> {pred_name}: {n_errors}")


# 31. FINAL SUMMARY

summary_no_bilstm = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Macro ROC-AUC",
        "Weighted ROC-AUC",
        "Macro AUPRC"
    ],
    "Value": [
        accuracy_no_bilstm,
        precision_macro_no_bilstm,
        recall_macro_no_bilstm,
        f1_macro_no_bilstm,
        f1_weighted_no_bilstm,
        macro_roc_auc_no_bilstm,
        weighted_roc_auc_no_bilstm,
        macro_auprc_no_bilstm
    ]
})

print("\n" + "=" * 70)
print("WITHOUT BiLSTM FINAL SUMMARY")
print("=" * 70)

print(
    summary_no_bilstm.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# 32. VARIABLES KEPT IN MEMORY

print("\n" + "=" * 70)
print("VARIABLES AVAILABLE FOR STATISTICS")
print("=" * 70)

print("""
MODEL:
    model_no_bilstm

TRAINING:
    history_no_bilstm

PREDICTIONS:
    y_pred_no_bilstm
    y_pred_prob_no_bilstm

CONFUSION MATRIX:
    cm_no_bilstm

ERROR ANALYSIS:
    error_df_no_bilstm

MAIN METRICS:
    accuracy_no_bilstm
    precision_macro_no_bilstm
    recall_macro_no_bilstm
    f1_macro_no_bilstm
    f1_weighted_no_bilstm

ROC-AUC:
    roc_auc_values_no_bilstm
    macro_roc_auc_no_bilstm
    weighted_roc_auc_no_bilstm

AUPRC:
    auprc_values_no_bilstm
    macro_auprc_no_bilstm

CONFIDENCE:
    max_prob_no_bilstm
    correct_mask_no_bilstm
    incorrect_mask_no_bilstm

SUMMARY:
    summary_no_bilstm

IMPORTANT:
    BiLSTM = REMOVED
    Attention = RETAINED
    Multi-scale CNN = RETAINED
""")

print("\nNo prediction files are saved.")
print("All required results remain available in memory.")

print("\n" + "=" * 70)
print("WITHOUT BiLSTM ABLATION COMPLETED")
print("=" * 70)

In [ ]:
# STATISTICAL ANALYSIS
# FULL MODEL vs WITHOUT BiLSTM

import numpy as np
import pandas as pd
import base64

from scipy.stats import binomtest


# 1. VERIFY VARIABLES AND ALIGNMENT

required_variables = [
    "y_test_full",
    "y_pred_full",
    "correctness_full",
    "y_test",
    "y_pred_no_bilstm"
]

for variable_name in required_variables:
    assert variable_name in globals(), (
        f"Variable missing: {variable_name}. "
        "Make sure the previous cells have been executed."
    )

y_true_stat = np.asarray(y_test_full).astype(int)
y_pred_full_stat = np.asarray(y_pred_full).astype(int)
y_pred_ablation_stat = np.asarray(y_pred_no_bilstm).astype(int)

assert np.array_equal(y_true_stat, np.asarray(y_test).astype(int)), (
    "The test labels of the full model and the ablation do not match."
)

assert len(y_true_stat) == len(y_pred_full_stat)
assert len(y_true_stat) == len(y_pred_ablation_stat)

assert np.array_equal(
    correctness_full,
    (y_pred_full_stat == y_true_stat).astype(np.uint8)
), "The correctness_full vector does not match the saved predictions."

correctness_ablation = (
    y_pred_ablation_stat == y_true_stat
).astype(np.uint8)

print("=" * 70)
print("STATISTICAL DATA VALIDATION")
print("=" * 70)

print("Number of test samples:", len(y_true_stat))
print("Full model accuracy:", correctness_full.mean())
print("Without BiLSTM accuracy:", correctness_ablation.mean())
print("Same test labels: YES")
print("Same number and order of predictions: verified by array alignment")


# 2. CORRECT / INCORRECT COUNTS

correct_full_count = int(correctness_full.sum())
incorrect_full_count = int(len(correctness_full) - correct_full_count)

correct_ablation_count = int(correctness_ablation.sum())
incorrect_ablation_count = int(
    len(correctness_ablation) - correct_ablation_count
)

print("\n" + "=" * 70)
print("CORRECTNESS COUNTS")
print("=" * 70)

print("FULL MODEL")
print("Correct:", correct_full_count)
print("Incorrect:", incorrect_full_count)
print("Accuracy:", f"{correctness_full.mean():.6f}")

print("\nWITHOUT BiLSTM")
print("Correct:", correct_ablation_count)
print("Incorrect:", incorrect_ablation_count)
print("Accuracy:", f"{correctness_ablation.mean():.6f}")


# 3. CLASS-WISE CORRECTNESS

class_names_stat = ["N", "S", "V"]

class_correctness_rows = []

for class_id, class_name in enumerate(class_names_stat):

    class_mask = (y_true_stat == class_id)
    n_class = int(class_mask.sum())

    full_correct = int(correctness_full[class_mask].sum())
    ablation_correct = int(correctness_ablation[class_mask].sum())

    full_accuracy_class = (
        full_correct / n_class if n_class > 0 else np.nan
    )

    ablation_accuracy_class = (
        ablation_correct / n_class if n_class > 0 else np.nan
    )

    class_correctness_rows.append({
        "Class": class_name,
        "Support": n_class,
        "Full correct": full_correct,
        "Full accuracy": full_accuracy_class,
        "Ablation correct": ablation_correct,
        "Ablation accuracy": ablation_accuracy_class
    })

class_correctness_df = pd.DataFrame(class_correctness_rows)

print("\n" + "=" * 70)
print("CLASS-WISE CORRECTNESS")
print("=" * 70)

print(
    class_correctness_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# 4. PACK CORRECTNESS VECTORS


correctness_full_packed = np.packbits(correctness_full)
correctness_ablation_packed = np.packbits(correctness_ablation)

correctness_full_base64 = base64.b64encode(
    correctness_full_packed.tobytes()
).decode("utf-8")

correctness_ablation_base64 = base64.b64encode(
    correctness_ablation_packed.tobytes()
).decode("utf-8")

original_vector_length = len(correctness_full)

print("\n" + "=" * 70)
print("PACKED CORRECTNESS VECTORS")
print("=" * 70)

print("Original vector length:", original_vector_length)
print("Full model packed bytes:", len(correctness_full_packed))
print("Ablation packed bytes:", len(correctness_ablation_packed))
print("Full model Base64 length:", len(correctness_full_base64))
print("Ablation Base64 length:", len(correctness_ablation_base64))


# 5. CONFUSION MATRIX CONSISTENCY CHECKS

cm_stat_ablation = confusion_matrix(
    y_true_stat,
    y_pred_ablation_stat,
    labels=[0, 1, 2]
)

print("\n" + "=" * 70)
print("ABLATION CONFUSION MATRIX VALIDATION")
print("=" * 70)

print(cm_stat_ablation)

assert cm_stat_ablation.sum() == len(y_true_stat)
assert np.trace(cm_stat_ablation) == correct_ablation_count
assert (cm_stat_ablation.sum() - np.trace(cm_stat_ablation)) == incorrect_ablation_count

accuracy_from_cm = (
    np.trace(cm_stat_ablation) / cm_stat_ablation.sum()
)

assert np.isclose(
    accuracy_from_cm,
    accuracy_score(y_true_stat, y_pred_ablation_stat)
)

print("Total matrix count:", cm_stat_ablation.sum())
print("Diagonal / correct:", np.trace(cm_stat_ablation))
print("Off-diagonal / incorrect:", cm_stat_ablation.sum() - np.trace(cm_stat_ablation))
print("Accuracy from matrix:", f"{accuracy_from_cm:.6f}")
print("Confusion matrix checks: PASSED")


# 6. PAIRED DISCORDANT PREDICTIONS


full_correct_ablation_wrong = int(
    np.sum((correctness_full == 1) & (correctness_ablation == 0))
)

full_wrong_ablation_correct = int(
    np.sum((correctness_full == 0) & (correctness_ablation == 1))
)

both_correct = int(
    np.sum((correctness_full == 1) & (correctness_ablation == 1))
)

both_wrong = int(
    np.sum((correctness_full == 0) & (correctness_ablation == 0))
)

print("\n" + "=" * 70)
print("PAIRED CORRECTNESS COMPARISON")
print("=" * 70)

print("Both models correct:", both_correct)
print("Both models incorrect:", both_wrong)
print("Full correct / Ablation incorrect:", full_correct_ablation_wrong)
print("Full incorrect / Ablation correct:", full_wrong_ablation_correct)


# 7. EXACT McNEMAR TEST


discordant_total = (
    full_correct_ablation_wrong
    + full_wrong_ablation_correct
)

if discordant_total == 0:

    mcnemar_p_value = 1.0
    print("\nNo discordant pairs: McNemar p-value = 1.000000")

else:

    mcnemar_result = binomtest(
        k=full_correct_ablation_wrong,
        n=discordant_total,
        p=0.5,
        alternative="two-sided"
    )

    mcnemar_p_value = mcnemar_result.pvalue

    print("\n" + "=" * 70)
    print("EXACT McNEMAR TEST")
    print("=" * 70)

    print("Discordant pairs:", discordant_total)
    print("Exact McNemar p-value:", f"{mcnemar_p_value:.8f}")

# 8. PAIRED T-TEST ON CORRECTNESS SCORES

from scipy.stats import ttest_rel

paired_ttest_result = ttest_rel(
    correctness_full.astype(float),
    correctness_ablation.astype(float)
)

paired_t_statistic = paired_ttest_result.statistic
paired_t_p_value = paired_ttest_result.pvalue

print("\n" + "=" * 70)
print("PAIRED T-TEST ON CORRECTNESS SCORES")
print("=" * 70)

print("Paired t-statistic:", f"{paired_t_statistic:.8f}")
print("Paired t-test p-value:", f"{paired_t_p_value:.8f}")


# 9. PAIRED BOOTSTRAP CI FOR ACCURACY DIFFERENCE


BOOTSTRAP_ITERATIONS = 10000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(BOOTSTRAP_SEED)

n_samples = len(y_true_stat)

observed_accuracy_difference = (
    correctness_full.mean() - correctness_ablation.mean()
)

bootstrap_differences = np.empty(
    BOOTSTRAP_ITERATIONS,
    dtype=float
)

for b in range(BOOTSTRAP_ITERATIONS):

    sampled_indices = rng.integers(
        0,
        n_samples,
        size=n_samples
    )

    sampled_full = correctness_full[sampled_indices]
    sampled_ablation = correctness_ablation[sampled_indices]

    bootstrap_differences[b] = (
        sampled_full.mean() - sampled_ablation.mean()
    )

ci_lower = np.percentile(bootstrap_differences, 2.5)
ci_upper = np.percentile(bootstrap_differences, 97.5)

print("\n" + "=" * 70)
print("PAIRED BOOTSTRAP - ACCURACY DIFFERENCE")
print("=" * 70)

print(
    "Observed accuracy difference (Full - Ablation):",
    f"{observed_accuracy_difference:.6f}"
)

print(
    "95% Bootstrap CI:",
    f"[{ci_lower:.6f}, {ci_upper:.6f}]"
)

print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Bootstrap seed:", BOOTSTRAP_SEED)


# 10. FINAL STATISTICAL SUMMARY

statistical_summary = pd.DataFrame({
    "Comparison": [
        "Full model",
        "Without BiLSTM",
        "Accuracy difference (Full - Ablation)",
        "McNemar exact p-value",
        "Bootstrap 95% CI lower",
        "Bootstrap 95% CI upper",
        "Paired t-test statistic",
        "Paired t-test p-value"
    ],
    "Value": [
        correctness_full.mean(),
        correctness_ablation.mean(),
        observed_accuracy_difference,
        mcnemar_p_value,
        ci_lower,
        ci_upper;paired_t_statistic,
        paired_t_p_value
    ]
})

print("\n" + "=" * 70)
print("FINAL STATISTICAL SUMMARY")
print("=" * 70)

print(
    statistical_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# 11. INTERPRETATION SUPPORT

print("\n" + "=" * 70)
print("STATISTICAL INTERPRETATION")
print("=" * 70)

alpha = 0.05

if mcnemar_p_value < alpha:
    print(
        "McNemar test: statistically significant difference "
        "in paired correctness at alpha = 0.05."
    )
else:
    print(
        "McNemar test: no statistically significant difference "
        "in paired correctness at alpha = 0.05."
    )

if ci_lower > 0:
    print(
        "Bootstrap CI: the interval is above zero; "
        "the observed accuracy difference favors the full model."
    )
elif ci_upper < 0:
    print(
        "Bootstrap CI: the interval is below zero; "
        "the observed accuracy difference favors the ablation."
    )
else:
    print(
        "Bootstrap CI: the interval includes zero; "
        "the accuracy difference is not clearly separated from zero."
    )

print("\nStatistical analysis completed.")

## Ablation without Attention

In [ ]:
#  PRESERVE FULL-MODEL PREDICTIONS

import numpy as np

y_test_full = np.asarray(y_test).copy()

y_pred_full = np.asarray(results["y_pred"]).copy()

# Store whether each full-model prediction is correct.
correctness_full = (y_pred_full == y_test_full).astype(np.uint8)

# Keep the full-model predictions and labels unchanged.
print("=" * 70)
print("FULL-MODEL PREDICTIONS PRESERVED")
print("=" * 70)

print("Number of test samples:", len(y_test_full))
print("Full-model prediction shape:", y_pred_full.shape)
print("Full-model correctness shape:", correctness_full.shape)
print("Full-model accuracy:", f"{correctness_full.mean():.6f}")

assert len(y_test_full) == len(y_pred_full), \
    "The full-model predictions and test labels have different lengths."

print("Full-model predictions are ready for paired statistical comparison.")

In [ ]:
# ABLATION - WITHOUT ATTENTION
# MS-CNN + BiLSTM

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    Concatenate,
    Bidirectional,
    LSTM
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)

from sklearn.preprocessing import label_binarize


# 1. REPRODUCIBILITY

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 70)
print("ABLATION - WITHOUT ATTENTION")
print("MS-CNN + BiLSTM")
print("=" * 70)

print("Random seed:", SEED)


# 2. LOAD SAME TRAIN / TEST DATA

DATA_DIR = "path working"

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Total samples:", len(X_train) + len(X_test))


# 3. VERIFY TEST SET AGAINST FULL MODEL

assert "y_test_full" in globals(), (
    "Cell preserve all full moel prediction has not been executed. "
    "Please preserve the full-model predictions first."
)

assert np.array_equal(y_test_full, y_test), (
    "WARNING: Test labels do not match "
    "the labels saved for the full model."
)

assert len(y_pred_full) == len(y_test), (
    "The number of full-model predictions differs "
    "from the number of test labels."
)

print("\nTest labels match the saved full-model labels.")


# 4. CLASS DISTRIBUTION

class_names = ["N", "S", "V"]
class_ids = [0, 1, 2]
num_classes = 3

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print("\nTraining:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_train == class_id)
    percentage = count / len(y_train) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")

print("\nTest:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_test == class_id)
    percentage = count / len(y_test) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")


# 5. RESHAPE

X_train = X_train.reshape(X_train.shape[0], 360, 1)
X_test = X_test.reshape(X_test.shape[0], 360, 1)

print("\n" + "=" * 70)
print("INPUT SHAPE")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


# 6. INPUT

inputs = Input(
    shape=(360, 1),
    name="ECG_Input"
)


# 7. MULTI-SCALE CNN

branch_3 = Conv1D(
    filters=32,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Conv1D_K3"
)(inputs)

branch_5 = Conv1D(
    filters=32,
    kernel_size=5,
    padding="same",
    activation="relu",
    name="Conv1D_K5"
)(inputs)

branch_7 = Conv1D(
    filters=32,
    kernel_size=7,
    padding="same",
    activation="relu",
    name="Conv1D_K7"
)(inputs)


# 8. CONCATENATION

x = Concatenate(
    axis=-1,
    name="MultiScale_Concatenation"
)([branch_3, branch_5, branch_7])


# 9. FEATURE REFINEMENT

x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Feature_Refinement_Conv"
)(x)

x = BatchNormalization(
    name="BatchNormalization"
)(x)

x = MaxPooling1D(
    pool_size=2,
    name="MaxPooling"
)(x)


# 10. BiLSTM

x = Bidirectional(
    LSTM(
        64,
        return_sequences=True
    ),
    name="BiLSTM"
)(x)


# 11. GLOBAL AVERAGE POOLING

x = GlobalAveragePooling1D(
    name="GlobalAveragePooling"
)(x)


# 12. CLASSIFIER

x = Dense(
    128,
    activation="relu",
    name="Dense_128"
)(x)

x = Dropout(
    0.5,
    name="Dropout_0.5"
)(x)

outputs = Dense(
    3,
    activation="softmax",
    name="Output_3_Classes"
)(x)


# 13. CREATE MODEL

model_no_attention = Model(
    inputs=inputs,
    outputs=outputs,
    name="MS_CNN_BiLSTM_Without_Attention"
)


# 14. ARCHITECTURE SUMMARY

print("\n" + "=" * 70)
print("WITHOUT ATTENTION ARCHITECTURE")
print("=" * 70)

model_no_attention.summary()

print("\nTotal parameters:", model_no_attention.count_params())


# 15. COMPILE

model_no_attention.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 16. TRAINING CONFIGURATION

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Epochs: 10")
print("Batch size: 128")
print("Validation split: 10%")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Dropout: 0.5")
print("Class weighting: None")
print("Oversampling: None")
print("Data augmentation: None")
print("BiLSTM: RETAINED")
print("Attention: REMOVED")


# 17. TRAIN

history_no_attention = model_no_attention.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.10,
    shuffle=True,
    verbose=1
)


# 18. TRAINING RESULTS

print("\n" + "=" * 70)
print("FINAL TRAINING / VALIDATION RESULTS")
print("=" * 70)

print(
    f"Final training accuracy: "
    f"{history_no_attention.history['accuracy'][-1]:.4f}"
)

print(
    f"Final validation accuracy: "
    f"{history_no_attention.history['val_accuracy'][-1]:.4f}"
)

print(
    f"Final training loss: "
    f"{history_no_attention.history['loss'][-1]:.4f}"
)

print(
    f"Final validation loss: "
    f"{history_no_attention.history['val_loss'][-1]:.4f}"
)


# 19. TRAINING CURVES

plt.figure(figsize=(8, 5))

plt.plot(
    history_no_attention.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_no_attention.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Without Attention - Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history_no_attention.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_no_attention.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Without Attention - Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 20. PREDICTIONS

print("\n" + "=" * 70)
print("PREDICTIONS")
print("=" * 70)

y_pred_prob_no_attention = model_no_attention.predict(
    X_test,
    batch_size=128,
    verbose=1
)

y_pred_no_attention = np.argmax(
    y_pred_prob_no_attention,
    axis=1
)

print("Prediction shape:", y_pred_no_attention.shape)
print("Probability shape:", y_pred_prob_no_attention.shape)


# 21. OVERALL METRICS

accuracy_no_attention = accuracy_score(
    y_test, y_pred_no_attention
)

precision_macro_no_attention = precision_score(
    y_test,
    y_pred_no_attention,
    average="macro",
    zero_division=0
)

recall_macro_no_attention = recall_score(
    y_test,
    y_pred_no_attention,
    average="macro",
    zero_division=0
)

f1_macro_no_attention = f1_score(
    y_test,
    y_pred_no_attention,
    average="macro",
    zero_division=0
)

f1_weighted_no_attention = f1_score(
    y_test,
    y_pred_no_attention,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("WITHOUT ATTENTION TEST RESULTS")
print("=" * 70)

print(f"Accuracy          : {accuracy_no_attention:.4f}")
print(f"Macro Precision   : {precision_macro_no_attention:.4f}")
print(f"Macro Recall      : {recall_macro_no_attention:.4f}")
print(f"Macro F1-score    : {f1_macro_no_attention:.4f}")
print(f"Weighted F1-score : {f1_weighted_no_attention:.4f}")


# 22. CLASSIFICATION REPORT

print("\n" + "=" * 70)
print("WITHOUT ATTENTION CLASSIFICATION REPORT")
print("=" * 70)

report_no_attention = classification_report(
    y_test,
    y_pred_no_attention,
    labels=class_ids,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report_no_attention)


# 23. CONFUSION MATRIX - RAW COUNTS

cm_no_attention = confusion_matrix(
    y_test,
    y_pred_no_attention,
    labels=class_ids
)

print("\n" + "=" * 70)
print("WITHOUT ATTENTION CONFUSION MATRIX - RAW COUNTS")
print("=" * 70)

print("\nRows = True Class")
print("Columns = Predicted Class\n")

cm_no_attention_df = pd.DataFrame(
    cm_no_attention,
    index=class_names,
    columns=class_names
)

print(cm_no_attention_df)
print("\nTotal test samples:", cm_no_attention.sum())


# 24. TP / FP / FN / TN

print("\n" + "=" * 70)
print("WITHOUT ATTENTION ONE-VS-REST ERROR ANALYSIS")
print("=" * 70)

total_test_no_attention = cm_no_attention.sum()
error_table_no_attention = []

for i, class_name in enumerate(class_names):

    TP = cm_no_attention[i, i]
    FN = cm_no_attention[i, :].sum() - TP
    FP = cm_no_attention[:, i].sum() - TP
    TN = total_test_no_attention - TP - FN - FP

    precision_class = (
        TP / (TP + FP) if (TP + FP) > 0 else 0
    )

    recall_class = (
        TP / (TP + FN) if (TP + FN) > 0 else 0
    )

    f1_class = (
        2 * precision_class * recall_class
        / (precision_class + recall_class)
        if (precision_class + recall_class) > 0
        else 0
    )

    error_table_no_attention.append([
        class_name,
        TP,
        FP,
        FN,
        TN,
        precision_class,
        recall_class,
        f1_class
    ])

    print(f"\nClass {class_name}")
    print("TP:", TP)
    print("FP:", FP)
    print("FN:", FN)
    print("TN:", TN)
    print(f"Precision: {precision_class:.4f}")
    print(f"Recall   : {recall_class:.4f}")
    print(f"F1-score : {f1_class:.4f}")


error_df_no_attention = pd.DataFrame(
    error_table_no_attention,
    columns=[
        "Class", "TP", "FP", "FN", "TN",
        "Precision", "Recall", "F1"
    ]
)

print("\n", error_df_no_attention.to_string(index=False))


# 25. CONFUSION MATRIX FIGURE

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm_no_attention,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    cbar=True
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Without Attention - Confusion Matrix")
plt.tight_layout()
plt.show()


# 26. ROC-AUC

print("\n" + "=" * 70)
print("WITHOUT ATTENTION ROC-AUC ANALYSIS")
print("=" * 70)

y_test_cat_no_attention = tf.keras.utils.to_categorical(
    y_test,
    num_classes=3
)

roc_auc_values_no_attention = {}

plt.figure(figsize=(8, 6))

for i, class_name in zip(class_ids, class_names):

    y_true_binary = (y_test == i).astype(int)

    fpr, tpr, _ = roc_curve(
        y_true_binary,
        y_pred_prob_no_attention[:, i]
    )

    roc_auc = auc(fpr, tpr)
    roc_auc_values_no_attention[class_name] = roc_auc

    print(f"{class_name} ROC-AUC: {roc_auc:.4f}")

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {roc_auc:.4f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Without Attention - ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_roc_auc_no_attention = roc_auc_score(
    y_test_cat_no_attention,
    y_pred_prob_no_attention,
    average="macro"
)

weighted_roc_auc_no_attention = roc_auc_score(
    y_test_cat_no_attention,
    y_pred_prob_no_attention,
    average="weighted"
)

print(f"\nMacro ROC-AUC    : {macro_roc_auc_no_attention:.4f}")
print(f"Weighted ROC-AUC : {weighted_roc_auc_no_attention:.4f}")


# 27. AUPRC

print("\n" + "=" * 70)
print("WITHOUT ATTENTION PRECISION-RECALL / AUPRC")
print("=" * 70)

auprc_values_no_attention = {}

plt.figure(figsize=(8, 6))

for i, class_name in zip(class_ids, class_names):

    y_true_binary = (y_test == i).astype(int)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_true_binary,
        y_pred_prob_no_attention[:, i]
    )

    ap = average_precision_score(
        y_true_binary,
        y_pred_prob_no_attention[:, i]
    )

    auprc_values_no_attention[class_name] = ap

    print(f"{class_name} AUPRC: {ap:.4f}")

    plt.plot(
        recall_curve,
        precision_curve,
        label=f"{class_name} (AP = {ap:.4f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Without Attention - Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_auprc_no_attention = np.mean(
    list(auprc_values_no_attention.values())
)

print(f"\nMacro AUPRC: {macro_auprc_no_attention:.4f}")


# 28. PREDICTION CONFIDENCE

max_prob_no_attention = np.max(
    y_pred_prob_no_attention,
    axis=1
)

correct_mask_no_attention = (
    y_pred_no_attention == y_test
)

incorrect_mask_no_attention = (
    y_pred_no_attention != y_test
)

print("\n" + "=" * 70)
print("WITHOUT ATTENTION PREDICTION CONFIDENCE")
print("=" * 70)

print(
    f"Mean confidence - correct: "
    f"{max_prob_no_attention[correct_mask_no_attention].mean():.4f}"
)

if incorrect_mask_no_attention.sum() > 0:
    print(
        f"Mean confidence - incorrect: "
        f"{max_prob_no_attention[incorrect_mask_no_attention].mean():.4f}"
    )
else:
    print("Mean confidence - incorrect: N/A (no incorrect predictions)")

print(f"Correct predictions: {correct_mask_no_attention.sum():,}")
print(f"Incorrect predictions: {incorrect_mask_no_attention.sum():,}")


# 29. CONFIDENCE BY CLASS

print("\n" + "=" * 70)
print("WITHOUT ATTENTION CONFIDENCE BY TRUE CLASS")
print("=" * 70)

for i, class_name in enumerate(class_names):

    class_mask = (y_test == i)

    print(
        f"{class_name}: "
        f"{max_prob_no_attention[class_mask].mean():.4f}"
    )


# 30. MISCLASSIFICATION ANALYSIS

print("\n" + "=" * 70)
print("WITHOUT ATTENTION MISCLASSIFICATION ANALYSIS")
print("=" * 70)

for true_id, true_name in zip(class_ids, class_names):

    print(f"\nTrue class: {true_name}")

    mask_true = (y_test == true_id)

    for pred_id, pred_name in zip(class_ids, class_names):

        if true_id != pred_id:

            n_errors = np.sum(
                mask_true & (y_pred_no_attention == pred_id)
            )

            if n_errors > 0:
                print(f"  {true_name} -> {pred_name}: {n_errors}")


# 31. FINAL SUMMARY

summary_no_attention = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Macro ROC-AUC",
        "Weighted ROC-AUC",
        "Macro AUPRC"
    ],
    "Value": [
        accuracy_no_attention,
        precision_macro_no_attention,
        recall_macro_no_attention,
        f1_macro_no_attention,
        f1_weighted_no_attention,
        macro_roc_auc_no_attention,
        weighted_roc_auc_no_attention,
        macro_auprc_no_attention
    ]
})

print("\n" + "=" * 70)
print("WITHOUT ATTENTION FINAL SUMMARY")
print("=" * 70)

print(
    summary_no_attention.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# 32. VARIABLES KEPT IN MEMORY

print("\n" + "=" * 70)
print("VARIABLES AVAILABLE FOR STATISTICS")
print("=" * 70)

print("""
MODEL:
    model_no_attention

TRAINING:
    history_no_attention

PREDICTIONS:
    y_pred_no_attention
    y_pred_prob_no_attention

CONFUSION MATRIX:
    cm_no_attention

ERROR ANALYSIS:
    error_df_no_attention

MAIN METRICS:
    accuracy_no_attention
    precision_macro_no_attention
    recall_macro_no_attention
    f1_macro_no_attention
    f1_weighted_no_attention

ROC-AUC:
    roc_auc_values_no_attention
    macro_roc_auc_no_attention
    weighted_roc_auc_no_attention

AUPRC:
    auprc_values_no_attention
    macro_auprc_no_attention

CONFIDENCE:
    max_prob_no_attention
    correct_mask_no_attention
    incorrect_mask_no_attention

SUMMARY:
    summary_no_attention

IMPORTANT:
    BiLSTM = RETAINED
    Attention = REMOVED
    Multi-scale CNN = RETAINED
""")

print("\nNo prediction files are saved.")
print("All required results remain available in memory.")

print("\n" + "=" * 70)
print("WITHOUT ATTENTION ABLATION COMPLETED")
print("=" * 70)

In [ ]:
# STATISTICAL ANALYSIS
# FULL MODEL vs WITHOUT ATTENTION

import numpy as np
import pandas as pd
import base64

from scipy.stats import binomtest


# 1. VERIFY VARIABLES AND ALIGNMENT

required_variables = [
    "y_test_full",
    "y_pred_full",
    "correctness_full",
    "y_test",
    "y_pred_no_attention"
]

for variable_name in required_variables:
    assert variable_name in globals(), (
        f"Variable missing: {variable_name}. "
        "Make sure the previous cells have been executed."
    )

y_true_stat = np.asarray(y_test_full).astype(int)
y_pred_full_stat = np.asarray(y_pred_full).astype(int)
y_pred_ablation_stat = np.asarray(y_pred_no_attention).astype(int)

assert np.array_equal(y_true_stat, np.asarray(y_test).astype(int)), (
    "The test labels of the full model and the ablation do not match."
)

assert len(y_true_stat) == len(y_pred_full_stat)
assert len(y_true_stat) == len(y_pred_ablation_stat)

assert np.array_equal(
    correctness_full,
    (y_pred_full_stat == y_true_stat).astype(np.uint8)
), "The correctness_full vector does not match the saved predictions."

correctness_ablation = (
    y_pred_ablation_stat == y_true_stat
).astype(np.uint8)

print("=" * 70)
print("STATISTICAL DATA VALIDATION")
print("=" * 70)

print("Number of test samples:", len(y_true_stat))
print("Full model accuracy:", correctness_full.mean())
print("Without Attention accuracy:", correctness_ablation.mean())
print("Same test labels: YES")
print("Same number and order of predictions: verified by array alignment")


# 2. CORRECT / INCORRECT COUNTS

correct_full_count = int(correctness_full.sum())
incorrect_full_count = int(len(correctness_full) - correct_full_count)

correct_ablation_count = int(correctness_ablation.sum())
incorrect_ablation_count = int(
    len(correctness_ablation) - correct_ablation_count
)

print("\n" + "=" * 70)
print("CORRECTNESS COUNTS")
print("=" * 70)

print("FULL MODEL")
print("Correct:", correct_full_count)
print("Incorrect:", incorrect_full_count)
print("Accuracy:", f"{correctness_full.mean():.6f}")

print("\nWITHOUT ATTENTION")
print("Correct:", correct_ablation_count)
print("Incorrect:", incorrect_ablation_count)
print("Accuracy:", f"{correctness_ablation.mean():.6f}")


# 3. CLASS-WISE CORRECTNESS

class_names_stat = ["N", "S", "V"]

class_correctness_rows = []

for class_id, class_name in enumerate(class_names_stat):

    class_mask = (y_true_stat == class_id)
    n_class = int(class_mask.sum())

    full_correct = int(correctness_full[class_mask].sum())
    ablation_correct = int(correctness_ablation[class_mask].sum())

    full_accuracy_class = (
        full_correct / n_class if n_class > 0 else np.nan
    )

    ablation_accuracy_class = (
        ablation_correct / n_class if n_class > 0 else np.nan
    )

    class_correctness_rows.append({
        "Class": class_name,
        "Support": n_class,
        "Full correct": full_correct,
        "Full accuracy": full_accuracy_class,
        "Ablation correct": ablation_correct,
        "Ablation accuracy": ablation_accuracy_class
    })

class_correctness_df = pd.DataFrame(class_correctness_rows)

print("\n" + "=" * 70)
print("CLASS-WISE CORRECTNESS")
print("=" * 70)

print(
    class_correctness_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# 4. PACK CORRECTNESS VECTORS


correctness_full_packed = np.packbits(correctness_full)
correctness_ablation_packed = np.packbits(correctness_ablation)

correctness_full_base64 = base64.b64encode(
    correctness_full_packed.tobytes()
).decode("utf-8")

correctness_ablation_base64 = base64.b64encode(
    correctness_ablation_packed.tobytes()
).decode("utf-8")

original_vector_length = len(correctness_full)

print("\n" + "=" * 70)
print("PACKED CORRECTNESS VECTORS")
print("=" * 70)

print("Original vector length:", original_vector_length)
print("Full model packed bytes:", len(correctness_full_packed))
print("Ablation packed bytes:", len(correctness_ablation_packed))
print("Full model Base64 length:", len(correctness_full_base64))
print("Ablation Base64 length:", len(correctness_ablation_base64))


# 5. CONFUSION MATRIX CONSISTENCY CHECKS

cm_stat_ablation = confusion_matrix(
    y_true_stat,
    y_pred_ablation_stat,
    labels=[0, 1, 2]
)

print("\n" + "=" * 70)
print("ABLATION CONFUSION MATRIX VALIDATION")
print("=" * 70)

print(cm_stat_ablation)

assert cm_stat_ablation.sum() == len(y_true_stat)
assert np.trace(cm_stat_ablation) == correct_ablation_count
assert (cm_stat_ablation.sum() - np.trace(cm_stat_ablation)) == incorrect_ablation_count

accuracy_from_cm = (
    np.trace(cm_stat_ablation) / cm_stat_ablation.sum()
)

assert np.isclose(
    accuracy_from_cm,
    accuracy_score(y_true_stat, y_pred_ablation_stat)
)

print("Total matrix count:", cm_stat_ablation.sum())
print("Diagonal / correct:", np.trace(cm_stat_ablation))
print("Off-diagonal / incorrect:", cm_stat_ablation.sum() - np.trace(cm_stat_ablation))
print("Accuracy from matrix:", f"{accuracy_from_cm:.6f}")
print("Confusion matrix checks: PASSED")


# 6. PAIRED DISCORDANT PREDICTIONS


full_correct_ablation_wrong = int(
    np.sum((correctness_full == 1) & (correctness_ablation == 0))
)

full_wrong_ablation_correct = int(
    np.sum((correctness_full == 0) & (correctness_ablation == 1))
)

both_correct = int(
    np.sum((correctness_full == 1) & (correctness_ablation == 1))
)

both_wrong = int(
    np.sum((correctness_full == 0) & (correctness_ablation == 0))
)

print("\n" + "=" * 70)
print("PAIRED CORRECTNESS COMPARISON")
print("=" * 70)

print("Both models correct:", both_correct)
print("Both models incorrect:", both_wrong)
print("Full correct / Ablation incorrect:", full_correct_ablation_wrong)
print("Full incorrect / Ablation correct:", full_wrong_ablation_correct)


# 7. EXACT McNEMAR TEST


discordant_total = (
    full_correct_ablation_wrong
    + full_wrong_ablation_correct
)

if discordant_total == 0:

    mcnemar_p_value = 1.0
    print("\nNo discordant pairs: McNemar p-value = 1.000000")

else:

    mcnemar_result = binomtest(
        k=full_correct_ablation_wrong,
        n=discordant_total,
        p=0.5,
        alternative="two-sided"
    )

    mcnemar_p_value = mcnemar_result.pvalue

    print("\n" + "=" * 70)
    print("EXACT McNEMAR TEST")
    print("=" * 70)

    print("Discordant pairs:", discordant_total)
    print("Exact McNemar p-value:", f"{mcnemar_p_value:.8f}")

# 8. PAIRED T-TEST ON CORRECTNESS SCORES

from scipy.stats import ttest_rel

paired_ttest_result = ttest_rel(
    correctness_full.astype(float),
    correctness_ablation.astype(float)
)

paired_t_statistic = paired_ttest_result.statistic
paired_t_p_value = paired_ttest_result.pvalue

print("\n" + "=" * 70)
print("PAIRED T-TEST ON CORRECTNESS SCORES")
print("=" * 70)

print("Paired t-statistic:", f"{paired_t_statistic:.8f}")
print("Paired t-test p-value:", f"{paired_t_p_value:.8f}")


# 9. PAIRED BOOTSTRAP CI FOR ACCURACY DIFFERENCE


BOOTSTRAP_ITERATIONS = 10000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(BOOTSTRAP_SEED)

n_samples = len(y_true_stat)

observed_accuracy_difference = (
    correctness_full.mean() - correctness_ablation.mean()
)

bootstrap_differences = np.empty(
    BOOTSTRAP_ITERATIONS,
    dtype=float
)

for b in range(BOOTSTRAP_ITERATIONS):

    sampled_indices = rng.integers(
        0,
        n_samples,
        size=n_samples
    )

    sampled_full = correctness_full[sampled_indices]
    sampled_ablation = correctness_ablation[sampled_indices]

    bootstrap_differences[b] = (
        sampled_full.mean() - sampled_ablation.mean()
    )

ci_lower = np.percentile(bootstrap_differences, 2.5)
ci_upper = np.percentile(bootstrap_differences, 97.5)

print("\n" + "=" * 70)
print("PAIRED BOOTSTRAP - ACCURACY DIFFERENCE")
print("=" * 70)

print(
    "Observed accuracy difference (Full - Ablation):",
    f"{observed_accuracy_difference:.6f}"
)

print(
    "95% Bootstrap CI:",
    f"[{ci_lower:.6f}, {ci_upper:.6f}]"
)

print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Bootstrap seed:", BOOTSTRAP_SEED)


# 10. FINAL STATISTICAL SUMMARY

statistical_summary_no_attention = pd.DataFrame({
    "Comparison": [
        "Full model",
        "Without Attention",
        "Accuracy difference (Full - Ablation)",
        "McNemar exact p-value",
        "Bootstrap 95% CI lower",
        "Bootstrap 95% CI upper",
        "Paired t-test statistic",
        "Paired t-test p-value"
    ],
    "Value": [
        correctness_full.mean(),
        correctness_ablation.mean(),
        observed_accuracy_difference,
        mcnemar_p_value,
        ci_lower,
        ci_upper;
        paired_t_statistic,
        paired_t_p_value
    ]
})

print("\n" + "=" * 70)
print("FINAL STATISTICAL SUMMARY")
print("=" * 70)

print(
    statistical_summary_no_attention.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# 11. INTERPRETATION SUPPORT

print("\n" + "=" * 70)
print("STATISTICAL INTERPRETATION")
print("=" * 70)

alpha = 0.05

if mcnemar_p_value < alpha:
    print(
        "McNemar test: statistically significant difference "
        "in paired correctness at alpha = 0.05."
    )
else:
    print(
        "McNemar test: no statistically significant difference "
        "in paired correctness at alpha = 0.05."
    )

if ci_lower > 0:
    print(
        "Bootstrap CI: the interval is above zero; "
        "the observed accuracy difference favors the full model."
    )
elif ci_upper < 0:
    print(
        "Bootstrap CI: the interval is below zero; "
        "the observed accuracy difference favors the ablation."
    )
else:
    print(
        "Bootstrap CI: the interval includes zero; "
        "the accuracy difference is not clearly separated from zero."
    )

print("\nStatistical analysis completed.")

## Ablation without multi scan CNN 

In [ ]:
#  PRESERVE FULL-MODEL PREDICTIONS

import numpy as np

y_test_full = np.asarray(y_test).copy()

y_pred_full = np.asarray(results["y_pred"]).copy()

# Store whether each full-model prediction is correct.
correctness_full = (y_pred_full == y_test_full).astype(np.uint8)

# Keep the full-model predictions and labels unchanged.
print("=" * 70)
print("FULL-MODEL PREDICTIONS PRESERVED")
print("=" * 70)

print("Number of test samples:", len(y_test_full))
print("Full-model prediction shape:", y_pred_full.shape)
print("Full-model correctness shape:", correctness_full.shape)
print("Full-model accuracy:", f"{correctness_full.mean():.6f}")

assert len(y_test_full) == len(y_pred_full), \
    "The full-model predictions and test labels have different lengths."

print("Full-model predictions are ready for paired statistical comparison.")

In [ ]:
# ABLATION - WITHOUT MCNN
# CNN + BiLSTM + Temporal Attention

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    Bidirectional,
    LSTM,
    Layer
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)

from sklearn.preprocessing import label_binarize


# 1. REPRODUCIBILITY

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 70)
print("ABLATION - WITHOUT MCNN")
print("CNN + BiLSTM + Temporal Attention")
print("=" * 70)

print("Random seed:", SEED)


# 2. LOAD SAME TRAIN / TEST DATA

DATA_DIR = "path working"

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Total samples:", len(X_train) + len(X_test))


# 3. VERIFY TEST SET AGAINST FULL MODEL

assert "y_test_full" in globals(), (
    "Cell preserve  full model predictions  has not been executed. "
    "Please preserve the full-model predictions first."
)

assert np.array_equal(y_test_full, y_test), (
    "WARNING: Test labels do not match "
    "the labels saved for the full model."
)

assert len(y_pred_full) == len(y_test), (
    "The number of full-model predictions differs "
    "from the number of test labels."
)

print("\nTest labels match the saved full-model labels.")


# 4. CLASS DISTRIBUTION

class_names = ["N", "S", "V"]
class_ids = [0, 1, 2]
num_classes = 3

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print("\nTraining:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_train == class_id)
    percentage = count / len(y_train) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")

print("\nTest:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_test == class_id)
    percentage = count / len(y_test) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")


# 5. RESHAPE

X_train = X_train.reshape(X_train.shape[0], 360, 1)
X_test = X_test.reshape(X_test.shape[0], 360, 1)

print("\n" + "=" * 70)
print("INPUT SHAPE")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


# 6. INPUT

inputs = Input(
    shape=(360, 1),
    name="ECG_Input"
)


# 7. STANDARD CNN

x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Standard_CNN_Conv"
)(inputs)


# 8. FEATURE REFINEMENT

x = BatchNormalization(
    name="BatchNormalization"
)(x)

x = MaxPooling1D(
    pool_size=2,
    name="MaxPooling"
)(x)


# 9. BiLSTM

x = Bidirectional(
    LSTM(
        64,
        return_sequences=True
    ),
    name="BiLSTM"
)(x)


# 10. TEMPORAL ATTENTION

class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            name="attention_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        scores = tf.tanh(tf.matmul(inputs, self.W) + self.b)
        attention_weights = tf.nn.softmax(scores, axis=1)

        context_vector = tf.reduce_sum(
            inputs * attention_weights,
            axis=1
        )

        return context_vector


x = AttentionLayer(
    name="Temporal_Attention"
)(x)


# 11. CLASSIFIER

x = Dense(
    128,
    activation="relu",
    name="Dense_128"
)(x)

x = Dropout(
    0.5,
    name="Dropout_0.5"
)(x)

outputs = Dense(
    3,
    activation="softmax",
    name="Output_3_Classes"
)(x)


# 12. CREATE MODEL

model_without_mcnn = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_BiLSTM_Attention_Without_MCNN"
)


# 13. ARCHITECTURE SUMMARY

print("\n" + "=" * 70)
print("WITHOUT MCNN ARCHITECTURE")
print("=" * 70)

model_without_mcnn.summary()

print("\nTotal parameters:", model_without_mcnn.count_params())


# 14. COMPILE

model_without_mcnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 15. TRAINING CONFIGURATION

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Epochs: 10")
print("Batch size: 128")
print("Validation split: 10%")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Dropout: 0.5")
print("Class weighting: None")
print("Oversampling: None")
print("Data augmentation: None")
print("BiLSTM: RETAINED")
print("Temporal Attention: RETAINED")
print("Multi-scale CNN: REMOVED")


# 16. TRAIN

history_without_mcnn = model_without_mcnn.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.10,
    shuffle=True,
    verbose=1
)


# 17. TRAINING RESULTS

print("\n" + "=" * 70)
print("FINAL TRAINING / VALIDATION RESULTS")
print("=" * 70)

print(
    f"Final training accuracy: "
    f"{history_without_mcnn.history['accuracy'][-1]:.4f}"
)

print(
    f"Final validation accuracy: "
    f"{history_without_mcnn.history['val_accuracy'][-1]:.4f}"
)

print(
    f"Final training loss: "
    f"{history_without_mcnn.history['loss'][-1]:.4f}"
)

print(
    f"Final validation loss: "
    f"{history_without_mcnn.history['val_loss'][-1]:.4f}"
)


# 18. TRAINING CURVES

plt.figure(figsize=(8, 5))

plt.plot(
    history_without_mcnn.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_without_mcnn.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Without MCNN - Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history_without_mcnn.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_without_mcnn.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Without MCNN - Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ABLATION - WITHOUT MCNN
# CNN + BiLSTM + Temporal Attention

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    Bidirectional,
    LSTM,
    Layer
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)

from sklearn.preprocessing import label_binarize


# 1. REPRODUCIBILITY

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 70)
print("ABLATION - WITHOUT MCNN")
print("CNN + BiLSTM + Temporal Attention")
print("=" * 70)

print("Random seed:", SEED)


# 2. LOAD SAME TRAIN / TEST DATA

DATA_DIR = "/kaggle/working"

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print("\n" + "=" * 70)
print("DATASET")
print("=" * 70)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Total samples:", len(X_train) + len(X_test))


# 3. VERIFY TEST SET AGAINST FULL MODEL

assert "y_test_full" in globals(), (
    "Cell 1 has not been executed. "
    "Please preserve the full-model predictions first."
)

assert np.array_equal(y_test_full, y_test), (
    "WARNING: Test labels do not match "
    "the labels saved for the full model."
)

assert len(y_pred_full) == len(y_test), (
    "The number of full-model predictions differs "
    "from the number of test labels."
)

print("\nTest labels match the saved full-model labels.")


# 4. CLASS DISTRIBUTION

class_names = ["N", "S", "V"]
class_ids = [0, 1, 2]
num_classes = 3

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print("\nTraining:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_train == class_id)
    percentage = count / len(y_train) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")

print("\nTest:")

for class_id, class_name in zip(class_ids, class_names):
    count = np.sum(y_test == class_id)
    percentage = count / len(y_test) * 100

    print(f"{class_name}: {count:,} ({percentage:.2f}%)")


# 5. RESHAPE

X_train = X_train.reshape(X_train.shape[0], 360, 1)
X_test = X_test.reshape(X_test.shape[0], 360, 1)

print("\n" + "=" * 70)
print("INPUT SHAPE")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


# 6. INPUT

inputs = Input(
    shape=(360, 1),
    name="ECG_Input"
)


# 7. STANDARD CNN

x = Conv1D(
    filters=64,
    kernel_size=3,
    padding="same",
    activation="relu",
    name="Standard_CNN_Conv"
)(inputs)


# 8. FEATURE REFINEMENT

x = BatchNormalization(
    name="BatchNormalization"
)(x)

x = MaxPooling1D(
    pool_size=2,
    name="MaxPooling"
)(x)


# 9. BiLSTM

x = Bidirectional(
    LSTM(
        64,
        return_sequences=True
    ),
    name="BiLSTM"
)(x)


# 10. TEMPORAL ATTENTION

class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b = self.add_weight(
            name="attention_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):
        scores = tf.tanh(tf.matmul(inputs, self.W) + self.b)
        attention_weights = tf.nn.softmax(scores, axis=1)

        context_vector = tf.reduce_sum(
            inputs * attention_weights,
            axis=1
        )

        return context_vector


x = AttentionLayer(
    name="Temporal_Attention"
)(x)


# 11. CLASSIFIER

x = Dense(
    128,
    activation="relu",
    name="Dense_128"
)(x)

x = Dropout(
    0.5,
    name="Dropout_0.5"
)(x)

outputs = Dense(
    3,
    activation="softmax",
    name="Output_3_Classes"
)(x)


# 12. CREATE MODEL

model_without_mcnn = Model(
    inputs=inputs,
    outputs=outputs,
    name="CNN_BiLSTM_Attention_Without_MCNN"
)


# 13. ARCHITECTURE SUMMARY

print("\n" + "=" * 70)
print("WITHOUT MCNN ARCHITECTURE")
print("=" * 70)

model_without_mcnn.summary()

print("\nTotal parameters:", model_without_mcnn.count_params())


# 14. COMPILE

model_without_mcnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# 15. TRAINING CONFIGURATION

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Epochs: 10")
print("Batch size: 128")
print("Validation split: 10%")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Dropout: 0.5")
print("Class weighting: None")
print("Oversampling: None")
print("Data augmentation: None")
print("BiLSTM: RETAINED")
print("Temporal Attention: RETAINED")
print("Multi-scale CNN: REMOVED")


# 16. TRAIN

history_without_mcnn = model_without_mcnn.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.10,
    shuffle=True,
    verbose=1
)


# 17. TRAINING RESULTS

print("\n" + "=" * 70)
print("FINAL TRAINING / VALIDATION RESULTS")
print("=" * 70)

print(
    f"Final training accuracy: "
    f"{history_without_mcnn.history['accuracy'][-1]:.4f}"
)

print(
    f"Final validation accuracy: "
    f"{history_without_mcnn.history['val_accuracy'][-1]:.4f}"
)

print(
    f"Final training loss: "
    f"{history_without_mcnn.history['loss'][-1]:.4f}"
)

print(
    f"Final validation loss: "
    f"{history_without_mcnn.history['val_loss'][-1]:.4f}"
)


# 18. TRAINING CURVES

plt.figure(figsize=(8, 5))

plt.plot(
    history_without_mcnn.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_without_mcnn.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Without MCNN - Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    history_without_mcnn.history["loss"],
    label="Training Loss"
)

plt.plot(
    history_without_mcnn.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Without MCNN - Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 20. PREDICTIONS

print("\n" + "=" * 70)
print("PREDICTIONS")
print("=" * 70)

y_pred_prob_without_mcnn = model_without_mcnn.predict(
    X_test,
    batch_size=128,
    verbose=1
)

y_pred_without_mcnn = np.argmax(
    y_pred_prob_without_mcnn,
    axis=1
)

print("Prediction shape:", y_pred_without_mcnn.shape)
print("Probability shape:", y_pred_prob_without_mcnn.shape)


# 21. OVERALL METRICS

accuracy_without_mcnn = accuracy_score(
    y_test, y_pred_without_mcnn
)

precision_macro_without_mcnn = precision_score(
    y_test,
    y_pred_without_mcnn,
    average="macro",
    zero_division=0
)

recall_macro_without_mcnn = recall_score(
    y_test,
    y_pred_without_mcnn,
    average="macro",
    zero_division=0
)

f1_macro_without_mcnn = f1_score(
    y_test,
    y_pred_without_mcnn,
    average="macro",
    zero_division=0
)

f1_weighted_without_mcnn = f1_score(
    y_test,
    y_pred_without_mcnn,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("WITHOUT MCNN TEST RESULTS")
print("=" * 70)

print(f"Accuracy          : {accuracy_without_mcnn:.4f}")
print(f"Macro Precision   : {precision_macro_without_mcnn:.4f}")
print(f"Macro Recall      : {recall_macro_without_mcnn:.4f}")
print(f"Macro F1-score    : {f1_macro_without_mcnn:.4f}")
print(f"Weighted F1-score : {f1_weighted_without_mcnn:.4f}")


# 22. CLASSIFICATION REPORT

print("\n" + "=" * 70)
print("WITHOUT MCNN CLASSIFICATION REPORT")
print("=" * 70)

report_without_mcnn = classification_report(
    y_test,
    y_pred_without_mcnn,
    labels=class_ids,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report_without_mcnn)


# 23. CONFUSION MATRIX - RAW COUNTS

cm_without_mcnn = confusion_matrix(
    y_test,
    y_pred_without_mcnn,
    labels=class_ids
)

print("\n" + "=" * 70)
print("WITHOUT MCNN CONFUSION MATRIX - RAW COUNTS")
print("=" * 70)

print("\nRows = True Class")
print("Columns = Predicted Class\n")

cm_without_mcnn_df = pd.DataFrame(
    cm_without_mcnn,
    index=class_names,
    columns=class_names
)

print(cm_without_mcnn_df)
print("\nTotal test samples:", cm_without_mcnn.sum())


# 24. TP / FP / FN / TN

print("\n" + "=" * 70)
print("WITHOUT MCNN ONE-VS-REST ERROR ANALYSIS")
print("=" * 70)

total_test_without_mcnn = cm_without_mcnn.sum()
error_table_without_mcnn = []

for i, class_name in enumerate(class_names):

    TP = cm_without_mcnn[i, i]
    FN = cm_without_mcnn[i, :].sum() - TP
    FP = cm_without_mcnn[:, i].sum() - TP
    TN = total_test_without_mcnn - TP - FN - FP

    precision_class = (
        TP / (TP + FP) if (TP + FP) > 0 else 0
    )

    recall_class = (
        TP / (TP + FN) if (TP + FN) > 0 else 0
    )

    f1_class = (
        2 * precision_class * recall_class
        / (precision_class + recall_class)
        if (precision_class + recall_class) > 0
        else 0
    )

    error_table_without_mcnn.append([
        class_name,
        TP,
        FP,
        FN,
        TN,
        precision_class,
        recall_class,
        f1_class
    ])

    print(f"\nClass {class_name}")
    print("TP:", TP)
    print("FP:", FP)
    print("FN:", FN)
    print("TN:", TN)
    print(f"Precision: {precision_class:.4f}")
    print(f"Recall   : {recall_class:.4f}")
    print(f"F1-score : {f1_class:.4f}")


error_df_without_mcnn = pd.DataFrame(
    error_table_without_mcnn,
    columns=[
        "Class", "TP", "FP", "FN", "TN",
        "Precision", "Recall", "F1"
    ]
)

print("\n", error_df_without_mcnn.to_string(index=False))


# 25. CONFUSION MATRIX FIGURE

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm_without_mcnn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    cbar=True
)

plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Without MCNN - Confusion Matrix")
plt.tight_layout()
plt.show()


# 26. ROC-AUC

print("\n" + "=" * 70)
print("WITHOUT MCNN ROC-AUC ANALYSIS")
print("=" * 70)

y_test_cat_without_mcnn = tf.keras.utils.to_categorical(
    y_test,
    num_classes=3
)

roc_auc_values_without_mcnn = {}

plt.figure(figsize=(8, 6))

for i, class_name in zip(class_ids, class_names):

    y_true_binary = (y_test == i).astype(int)

    fpr, tpr, _ = roc_curve(
        y_true_binary,
        y_pred_prob_without_mcnn[:, i]
    )

    roc_auc = auc(fpr, tpr)
    roc_auc_values_without_mcnn[class_name] = roc_auc

    print(f"{class_name} ROC-AUC: {roc_auc:.4f}")

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {roc_auc:.4f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Without MCNN - ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_roc_auc_without_mcnn = roc_auc_score(
    y_test_cat_without_mcnn,
    y_pred_prob_without_mcnn,
    average="macro"
)

weighted_roc_auc_without_mcnn = roc_auc_score(
    y_test_cat_without_mcnn,
    y_pred_prob_without_mcnn,
    average="weighted"
)

print(f"\nMacro ROC-AUC    : {macro_roc_auc_without_mcnn:.4f}")
print(f"Weighted ROC-AUC : {weighted_roc_auc_without_mcnn:.4f}")


# 27. AUPRC

print("\n" + "=" * 70)
print("WITHOUT MCNN PRECISION-RECALL / AUPRC")
print("=" * 70)

auprc_values_without_mcnn = {}

plt.figure(figsize=(8, 6))

for i, class_name in zip(class_ids, class_names):

    y_true_binary = (y_test == i).astype(int)

    precision_curve, recall_curve, _ = precision_recall_curve(
        y_true_binary,
        y_pred_prob_without_mcnn[:, i]
    )

    ap = average_precision_score(
        y_true_binary,
        y_pred_prob_without_mcnn[:, i]
    )

    auprc_values_without_mcnn[class_name] = ap

    print(f"{class_name} AUPRC: {ap:.4f}")

    plt.plot(
        recall_curve,
        precision_curve,
        label=f"{class_name} (AP = {ap:.4f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Without MCNN - Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_auprc_without_mcnn = np.mean(
    list(auprc_values_without_mcnn.values())
)

print(f"\nMacro AUPRC: {macro_auprc_without_mcnn:.4f}")


# 28. PREDICTION CONFIDENCE

max_prob_without_mcnn = np.max(
    y_pred_prob_without_mcnn,
    axis=1
)

correct_mask_without_mcnn = (
    y_pred_without_mcnn == y_test
)

incorrect_mask_without_mcnn = (
    y_pred_without_mcnn != y_test
)

print("\n" + "=" * 70)
print("WITHOUT MCNN PREDICTION CONFIDENCE")
print("=" * 70)

print(
    f"Mean confidence - correct: "
    f"{max_prob_without_mcnn[correct_mask_without_mcnn].mean():.4f}"
)

if incorrect_mask_without_mcnn.sum() > 0:
    print(
        f"Mean confidence - incorrect: "
        f"{max_prob_without_mcnn[incorrect_mask_without_mcnn].mean():.4f}"
    )
else:
    print("Mean confidence - incorrect: N/A (no incorrect predictions)")

print(f"Correct predictions: {correct_mask_without_mcnn.sum():,}")
print(f"Incorrect predictions: {incorrect_mask_without_mcnn.sum():,}")


# 29. CONFIDENCE BY CLASS

print("\n" + "=" * 70)
print("WITHOUT MCNN CONFIDENCE BY TRUE CLASS")
print("=" * 70)

for i, class_name in enumerate(class_names):

    class_mask = (y_test == i)

    print(
        f"{class_name}: "
        f"{max_prob_without_mcnn[class_mask].mean():.4f}"
    )


# 30. MISCLASSIFICATION ANALYSIS

print("\n" + "=" * 70)
print("WITHOUT MCNN MISCLASSIFICATION ANALYSIS")
print("=" * 70)

for true_id, true_name in zip(class_ids, class_names):

    print(f"\nTrue class: {true_name}")

    mask_true = (y_test == true_id)

    for pred_id, pred_name in zip(class_ids, class_names):

        if true_id != pred_id:

            n_errors = np.sum(
                mask_true & (y_pred_without_mcnn == pred_id)
            )

            if n_errors > 0:
                print(f"  {true_name} -> {pred_name}: {n_errors}")


# 31. FINAL SUMMARY

summary_without_mcnn = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Macro ROC-AUC",
        "Weighted ROC-AUC",
        "Macro AUPRC"
    ],
    "Value": [
        accuracy_without_mcnn,
        precision_macro_without_mcnn,
        recall_macro_without_mcnn,
        f1_macro_without_mcnn,
        f1_weighted_without_mcnn,
        macro_roc_auc_without_mcnn,
        weighted_roc_auc_without_mcnn,
        macro_auprc_without_mcnn
    ]
})

print("\n" + "=" * 70)
print("WITHOUT MCNN FINAL SUMMARY")
print("=" * 70)

print(
    summary_without_mcnn.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# 32. VARIABLES KEPT IN MEMORY

print("\n" + "=" * 70)
print("VARIABLES AVAILABLE FOR STATISTICS")
print("=" * 70)

print("""
MODEL:
    model_without_mcnn

TRAINING:
    history_without_mcnn

PREDICTIONS:
    y_pred_without_mcnn
    y_pred_prob_without_mcnn

CONFUSION MATRIX:
    cm_without_mcnn

ERROR ANALYSIS:
    error_df_without_mcnn

MAIN METRICS:
    accuracy_without_mcnn
    precision_macro_without_mcnn
    recall_macro_without_mcnn
    f1_macro_without_mcnn
    f1_weighted_without_mcnn

ROC-AUC:
    roc_auc_values_without_mcnn
    macro_roc_auc_without_mcnn
    weighted_roc_auc_without_mcnn

AUPRC:
    auprc_values_without_mcnn
    macro_auprc_without_mcnn

CONFIDENCE:
    max_prob_without_mcnn
    correct_mask_without_mcnn
    incorrect_mask_without_mcnn

SUMMARY:
    summary_without_mcnn

IMPORTANT:
    BiLSTM = RETAINED
    Temporal Attention = RETAINED
    Multi-scale CNN = REMOVED
""")

print("\nNo prediction files are saved.")
print("All required results remain available in memory.")

print("\n" + "=" * 70)
print("WITHOUT MCNN ABLATION COMPLETED")
print("=" * 70)

In [ ]:
# STATISTICAL ANALYSIS
# FULL MODEL vs WITHOUT MCNN

import numpy as np
import pandas as pd
import base64

from scipy.stats import binomtest


# 1. VERIFY VARIABLES AND ALIGNMENT

required_variables = [
    "y_test_full",
    "y_pred_full",
    "correctness_full",
    "y_test",
    "y_pred_without_mcnn"
]

for variable_name in required_variables:
    assert variable_name in globals(), (
        f"Variable missing: {variable_name}. "
        "Make sure the previous cells have been executed."
    )

y_true_stat = np.asarray(y_test_full).astype(int)
y_pred_full_stat = np.asarray(y_pred_full).astype(int)
y_pred_ablation_stat = np.asarray(y_pred_without_mcnn).astype(int)

assert np.array_equal(y_true_stat, np.asarray(y_test).astype(int)), (
    "The test labels of the full model and the ablation do not match."
)

assert len(y_true_stat) == len(y_pred_full_stat)
assert len(y_true_stat) == len(y_pred_ablation_stat)

assert np.array_equal(
    correctness_full,
    (y_pred_full_stat == y_true_stat).astype(np.uint8)
), "The correctness_full vector does not match the saved predictions."

correctness_ablation = (
    y_pred_ablation_stat == y_true_stat
).astype(np.uint8)

print("=" * 70)
print("STATISTICAL DATA VALIDATION")
print("=" * 70)

print("Number of test samples:", len(y_true_stat))
print("Full model accuracy:", correctness_full.mean())
print("Without MCNN accuracy:", correctness_ablation.mean())
print("Same test labels: YES")
print("Same number and order of predictions: verified by array alignment")


# 2. CORRECT / INCORRECT COUNTS

correct_full_count = int(correctness_full.sum())
incorrect_full_count = int(len(correctness_full) - correct_full_count)

correct_ablation_count = int(correctness_ablation.sum())
incorrect_ablation_count = int(
    len(correctness_ablation) - correct_ablation_count
)

print("\n" + "=" * 70)
print("CORRECTNESS COUNTS")
print("=" * 70)

print("FULL MODEL")
print("Correct:", correct_full_count)
print("Incorrect:", incorrect_full_count)
print("Accuracy:", f"{correctness_full.mean():.6f}")

print("\nWITHOUT MCNN")
print("Correct:", correct_ablation_count)
print("Incorrect:", incorrect_ablation_count)
print("Accuracy:", f"{correctness_ablation.mean():.6f}")


# 3. CLASS-WISE CORRECTNESS

class_names_stat = ["N", "S", "V"]

class_correctness_rows = []

for class_id, class_name in enumerate(class_names_stat):

    class_mask = (y_true_stat == class_id)
    n_class = int(class_mask.sum())

    full_correct = int(correctness_full[class_mask].sum())
    ablation_correct = int(correctness_ablation[class_mask].sum())

    full_accuracy_class = (
        full_correct / n_class if n_class > 0 else np.nan
    )

    ablation_accuracy_class = (
        ablation_correct / n_class if n_class > 0 else np.nan
    )

    class_correctness_rows.append({
        "Class": class_name,
        "Support": n_class,
        "Full correct": full_correct,
        "Full accuracy": full_accuracy_class,
        "Ablation correct": ablation_correct,
        "Ablation accuracy": ablation_accuracy_class
    })

class_correctness_df = pd.DataFrame(class_correctness_rows)

print("\n" + "=" * 70)
print("CLASS-WISE CORRECTNESS")
print("=" * 70)

print(
    class_correctness_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# 4. PACK CORRECTNESS VECTORS


correctness_full_packed = np.packbits(correctness_full)
correctness_ablation_packed = np.packbits(correctness_ablation)

correctness_full_base64 = base64.b64encode(
    correctness_full_packed.tobytes()
).decode("utf-8")

correctness_ablation_base64 = base64.b64encode(
    correctness_ablation_packed.tobytes()
).decode("utf-8")

original_vector_length = len(correctness_full)

print("\n" + "=" * 70)
print("PACKED CORRECTNESS VECTORS")
print("=" * 70)

print("Original vector length:", original_vector_length)
print("Full model packed bytes:", len(correctness_full_packed))
print("Ablation packed bytes:", len(correctness_ablation_packed))
print("Full model Base64 length:", len(correctness_full_base64))
print("Ablation Base64 length:", len(correctness_ablation_base64))


# 5. CONFUSION MATRIX CONSISTENCY CHECKS

cm_stat_ablation = confusion_matrix(
    y_true_stat,
    y_pred_ablation_stat,
    labels=[0, 1, 2]
)

print("\n" + "=" * 70)
print("ABLATION CONFUSION MATRIX VALIDATION")
print("=" * 70)

print(cm_stat_ablation)

assert cm_stat_ablation.sum() == len(y_true_stat)
assert np.trace(cm_stat_ablation) == correct_ablation_count
assert (cm_stat_ablation.sum() - np.trace(cm_stat_ablation)) == incorrect_ablation_count

accuracy_from_cm = (
    np.trace(cm_stat_ablation) / cm_stat_ablation.sum()
)

assert np.isclose(
    accuracy_from_cm,
    accuracy_score(y_true_stat, y_pred_ablation_stat)
)

print("Total matrix count:", cm_stat_ablation.sum())
print("Diagonal / correct:", np.trace(cm_stat_ablation))
print("Off-diagonal / incorrect:", cm_stat_ablation.sum() - np.trace(cm_stat_ablation))
print("Accuracy from matrix:", f"{accuracy_from_cm:.6f}")
print("Confusion matrix checks: PASSED")


# 6. PAIRED DISCORDANT PREDICTIONS


full_correct_ablation_wrong = int(
    np.sum((correctness_full == 1) & (correctness_ablation == 0))
)

full_wrong_ablation_correct = int(
    np.sum((correctness_full == 0) & (correctness_ablation == 1))
)

both_correct = int(
    np.sum((correctness_full == 1) & (correctness_ablation == 1))
)

both_wrong = int(
    np.sum((correctness_full == 0) & (correctness_ablation == 0))
)

print("\n" + "=" * 70)
print("PAIRED CORRECTNESS COMPARISON")
print("=" * 70)

print("Both models correct:", both_correct)
print("Both models incorrect:", both_wrong)
print("Full correct / Ablation incorrect:", full_correct_ablation_wrong)
print("Full incorrect / Ablation correct:", full_wrong_ablation_correct)


# 7. EXACT McNEMAR TEST


discordant_total = (
    full_correct_ablation_wrong
    + full_wrong_ablation_correct
)

if discordant_total == 0:

    mcnemar_p_value = 1.0
    print("\nNo discordant pairs: McNemar p-value = 1.000000")

else:

    mcnemar_result = binomtest(
        k=full_correct_ablation_wrong,
        n=discordant_total,
        p=0.5,
        alternative="two-sided"
    )

    mcnemar_p_value = mcnemar_result.pvalue

    print("\n" + "=" * 70)
    print("EXACT McNEMAR TEST")
    print("=" * 70)

    print("Discordant pairs:", discordant_total)
    print("Exact McNemar p-value:", f"{mcnemar_p_value:.8f}")
    
# 8. PAIRED T-TEST ON CORRECTNESS SCORES

from scipy.stats import ttest_rel

paired_ttest_result = ttest_rel(
    correctness_full.astype(float),
    correctness_ablation.astype(float)
)

paired_t_statistic = paired_ttest_result.statistic
paired_t_p_value = paired_ttest_result.pvalue

print("\n" + "=" * 70)
print("PAIRED T-TEST ON CORRECTNESS SCORES")
print("=" * 70)

print("Paired t-statistic:", f"{paired_t_statistic:.8f}")
print("Paired t-test p-value:", f"{paired_t_p_value:.8f}")


# 9. PAIRED BOOTSTRAP CI FOR ACCURACY DIFFERENCE


BOOTSTRAP_ITERATIONS = 10000
BOOTSTRAP_SEED = 42

rng = np.random.default_rng(BOOTSTRAP_SEED)

n_samples = len(y_true_stat)

observed_accuracy_difference = (
    correctness_full.mean() - correctness_ablation.mean()
)

bootstrap_differences = np.empty(
    BOOTSTRAP_ITERATIONS,
    dtype=float
)

for b in range(BOOTSTRAP_ITERATIONS):

    sampled_indices = rng.integers(
        0,
        n_samples,
        size=n_samples
    )

    sampled_full = correctness_full[sampled_indices]
    sampled_ablation = correctness_ablation[sampled_indices]

    bootstrap_differences[b] = (
        sampled_full.mean() - sampled_ablation.mean()
    )

ci_lower = np.percentile(bootstrap_differences, 2.5)
ci_upper = np.percentile(bootstrap_differences, 97.5)

print("\n" + "=" * 70)
print("PAIRED BOOTSTRAP - ACCURACY DIFFERENCE")
print("=" * 70)

print(
    "Observed accuracy difference (Full - Ablation):",
    f"{observed_accuracy_difference:.6f}"
)

print(
    "95% Bootstrap CI:",
    f"[{ci_lower:.6f}, {ci_upper:.6f}]"
)

print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Bootstrap seed:", BOOTSTRAP_SEED)


# 10. FINAL STATISTICAL SUMMARY

statistical_summary_without_mcnn = pd.DataFrame({
    "Comparison": [
        "Full model",
        "Without MCNN",
        "Accuracy difference (Full - Ablation)",
        "McNemar exact p-value",
        "Bootstrap 95% CI lower",
        "Bootstrap 95% CI upper";
        "Paired t-test statistic",
        "Paired t-test p-value"
    ],
    "Value": [
        correctness_full.mean(),
        correctness_ablation.mean(),
        observed_accuracy_difference,
        mcnemar_p_value,
        ci_lower,
        ci_upper;
        paired_t_statistic,
        paired_t_p_value
    ]
})

print("\n" + "=" * 70)
print("FINAL STATISTICAL SUMMARY")
print("=" * 70)

print(
    statistical_summary_without_mcnn.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# 10. INTERPRETATION SUPPORT

print("\n" + "=" * 70)
print("STATISTICAL INTERPRETATION")
print("=" * 70)

alpha = 0.05

if mcnemar_p_value < alpha:
    print(
        "McNemar test: statistically significant difference "
        "in paired correctness at alpha = 0.05."
    )
else:
    print(
        "McNemar test: no statistically significant difference "
        "in paired correctness at alpha = 0.05."
    )

if ci_lower > 0:
    print(
        "Bootstrap CI: the interval is above zero; "
        "the observed accuracy difference favors the full model."
    )
elif ci_upper < 0:
    print(
        "Bootstrap CI: the interval is below zero; "
        "the observed accuracy difference favors the ablation."
    )
else:
    print(
        "Bootstrap CI: the interval includes zero; "
        "the accuracy difference is not clearly separated from zero."
    )

print("\nStatistical analysis completed.")